In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

In [ ]:
matplotlib.rcParams['savefig.dpi'] = 400  # Set the DPI for saving figures
matplotlib.rcParams['axes.labelweight'] = 'bold'  # Bold x and y labels
matplotlib.rcParams['axes.labelsize'] = 25  # Maximize font size of x and y labels
# matplotlib.rcParams['axes.grid'] = True  # Enable grid view
matplotlib.rcParams['axes.edgecolor'] = 'black'  # Border color
matplotlib.rcParams['axes.linewidth'] = 2  # Border thickness
# tight layout
matplotlib.rcParams['figure.autolayout'] = True

In [ ]:
dataset = pd.read_csv("<input csv file>")


In [ ]:
# Remove rows where vpn starts with package
dataset = dataset[~dataset.apply(lambda row: str(row['vpn']).startswith(row['package']) if pd.notna(row['vpn']) else False, axis=1)]
dataset = dataset[~dataset.apply(lambda row: str(row['tor_bridge']).startswith(row['package']) if pd.notna(row['tor_bridge']) else False, axis=1)]


In [ ]:
# remove rows where package starts with org.torproject
dataset = dataset[~dataset['package'].str.startswith('org.torproject')]

In [ ]:
dataset.shape

In [ ]:
# for any row where lower of tor_bridge has shadowsocks in it and vpn is null, replace vpn with dataset["tor_bridge"]
dataset.loc[dataset['tor_bridge'].str.lower().str.contains('shadowsocks') & dataset['vpn'].isnull(), 'vpn'] = dataset['tor_bridge']
# make the tor bridge column with shadowsocks empty
dataset.loc[
    dataset['tor_bridge'].fillna('').str.lower().str.contains('shadowsocks'), 
    'tor_bridge'
] = ''


In [ ]:
# remove any row with duplicate hash
dataset = dataset.drop_duplicates(subset='hash', keep='first')

In [ ]:
dataset.loc[dataset["tor"] == "r/tOr", "tor"] = np.nan


In [ ]:
    # Count distinct packages in descending order
package_counts = dataset['package'].value_counts().reset_index()
package_counts.columns = ['package', 'count']

# Determine categories efficiently
category_mapping = dataset.groupby('package').agg({
    'tor': lambda x: 'tor' if x.notna().any() else None,
    'tor_bridge': lambda x: 'tor_bridge' if x.notna().any() else None,
    'proxy': lambda x: 'proxy' if x.notna().any() else None,
    'vpn': lambda x: 'vpn' if x.notna().any() else None,
    'i2p' : lambda x: 'i2p' if x.notna().any() else None,
}).fillna('')

# Combine the categories into a single string
category_mapping['category'] = category_mapping[['tor', 'tor_bridge', 'proxy', 'vpn', 'i2p']].apply(
    lambda x: ', '.join(filter(None, x)), axis=1
)

# Merge the categories with the package counts
package_counts = package_counts.merge(category_mapping[['category']], left_on='package', right_index=True)

print(package_counts)

In [ ]:
package_counts.to_csv("<input csv file>", index=False)

In [ ]:
package_counts = pd.read_csv("<input csv file>")

In [ ]:
plt.hist

In [ ]:
row = dataset[dataset["vpn"].notna()].loc[dataset[dataset["vpn"].notna()]['score'].idxmax()]
print(row["package"])
print(row["user"])
print(row["tor"])
print(row["path"])
print(row["score"])

In [ ]:
                # add to the dataset, rows of dataset1 where vpn is not null and hash is not in dataset
dataset = pd.concat([dataset, dataset1[dataset1['vpn'].notnull() & ~dataset1['hash'].isin(dataset['hash'])]])
dataset = pd.concat([dataset, dataset1[dataset1['tor_bridge'].notnull() & ~dataset1['hash'].isin(dataset['hash'])]])

In [ ]:
dataset.shape       

In [ ]:
dataset["vpn"].value_counts()

In [ ]:
dataset1["vpn"].value_counts()

In [ ]:
dataset["min_sdk_version"].max()

In [ ]:
dataset["min_sdk_version"].value_counts()

In [ ]:
dataset["target_sdk_version"].value_counts()

In [ ]:
dataset.to_csv("<path-to-directory>", index=False)

In [ ]:
# year_totals_malware = {
#     2009: 0,
#     2010: 0,
#     2011: 32,
#     2012: 9970,
#     2013: 219123,
#     2014: 541250,
#     2015: 131981,
#     2016: 606736,
#     2017: 123704,
#     2018: 350343,
#     2019: 432544,
#     2020: 269584,
#     2021: 426224,
#     2022: 424749,
#     2023: 88738,
#     2024: 25399
# }
year_totals_malware = {
    2009: 0,
    2010: 0,
    2011: 32,
    2012: 9970,
    2013: 219123,
    2014: 541250,
    2015: 131981,
    2016: 606736,
    2017: 123704,
    2018: 350343,
    2019: 432544,
    2020: 269584,
    2021: 426224,
    2022: 424749,
    2023: 88738,
    2024: 25399,
    2025: 2455
}

In [ ]:
len(set(dataset["vpn"]))

In [ ]:
def make_heatmap(pivot_table, filename, fig_size=(40,15), take_last=False):
    fig = plt.figure(figsize=fig_size)
    # gs = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[0.08, 0.0002, 1.5])  # Allocate 10% height for the colorbar and the rest for the heatmap
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.08, 1.5])

    # Create the axis for the colorbar
    cbar_ax = fig.add_subplot(gs[0])  # Top axis for the colorbar

    # Create the axis for the heatmap
    heatmap_ax = fig.add_subplot(gs[-1])  # Bottom axis for the heatmap

    # Plot the heatmap
    heatmap = sns.heatmap(
        pivot_table,
        ax=heatmap_ax,
        cmap='Blues',           # Cyan color palette
        linewidths=0.5,         # Add spacing between boxes
        linecolor='lightgrey',  # Light grid color
        annot=True,             # Display counts within boxes
        fmt='g',                # Format annotations as integers
        annot_kws={"fontsize": 14, "rotation": 45, "fontweight": 'bold'},  # Rotate annotations diagonally and increase font size
        cbar_ax=cbar_ax,        # Use the custom colorbar axis
        cbar_kws={'orientation': 'horizontal'}  # Set colorbar orientation to horizontal
    )

    for spine in heatmap_ax.spines.values():
        spine.set_visible(True)      # Make the spines visible
        spine.set_color('black')     # Set border color
        spine.set_linewidth(2)       # Set border width

    # Customize the horizontal colorbar
    colorbar = heatmap.collections[0].colorbar  # Access the color bar
    max_val = int(colorbar.vmax)  # Maximum value of the color bar
    if not take_last:
        tick_positions = np.concatenate([np.arange(0, max_val + 10000, 10000)[:-1], [max_val]])  # Tick positions with 10K intervals
    else:
        tick_positions = np.arange(0, max_val + 10000, 10000)[:-1]
    colorbar.set_ticks(tick_positions)  # Apply ticks
    if max_val > 50000:
        colorbar.set_ticklabels([f"{int(val / 1000)}K" for val in tick_positions], fontweight="bold", fontsize=32)  # Format tick labels
    colorbar.set_label(
        "Counts for Statically Validated APKs", 
        fontsize=20, fontweight='bold', labelpad=10  # Add label to the colorbar
    )

    # Increase font size of color bar ticks
    colorbar.ax.tick_params(labelsize=22)

    # Customize heatmap axes
    # heatmap_ax.set_xticks(heatmap_ax.get_xticks())
    # heatmap_ax.set_xticklabels(heatmap_ax.get_xticklabels(), fontsize=18, fontweight='bold')
    xticks = heatmap_ax.get_xticks()
    heatmap_ax.set_xticks(xticks)
    heatmap_ax.set_xticklabels([int(tick) for tick in xticks], fontsize=18, fontweight='bold')
    heatmap_ax.set_yticklabels(heatmap_ax.get_yticklabels(), fontsize=26, fontweight='bold', rotation=0)
    heatmap_ax.set_xlabel("Virus Total Score", fontsize=26, fontweight='bold')
    heatmap_ax.set_ylabel("Year", fontsize=30, fontweight='bold')

    # Customize spines
    for spine in heatmap_ax.spines.values():
        spine.set_linewidth(2)
        spine.set_color('black')

    # Save and show the plot
    plt.savefig(f"<output directory>/{filename}.png", bbox_inches='tight')
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Create the pivot table
pivot_table = dataset.pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

pivot_table['Total'] = pivot_table.sum(axis=1)

# Add a "Total" row
pivot_table.loc['Total'] = pivot_table.sum(axis=0)

make_heatmap(pivot_table, "heatmap_score")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Filter dataset for years 2020 to 2025 inclusive
filtered_dataset = dataset[(dataset['year'] >= 2020) & (dataset['year'] <= 2025)]

# Create the pivot table from the filtered dataset
pivot_table = filtered_dataset.pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

# Optionally, add a Total column and row if needed
pivot_table['Total'] = pivot_table.sum(axis=1)
pivot_table.loc['Total'] = pivot_table.sum(axis=0)

# Generate the heatmap

make_heatmap(pivot_table, "heatmap_score_trunc", fig_size=(40,8))


In [ ]:
pivot_table_vpn = dataset[dataset['vpn'].notna() & (dataset['vpn'] != '')].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

pivot_table_vpn['Total'] = pivot_table_vpn.sum(axis=1)

# Add a "Total" row
pivot_table_vpn.loc['Total'] = pivot_table_vpn.sum(axis=0)

make_heatmap(pivot_table_vpn, "heatmap_score_vpn")

In [ ]:
pivot_table_tor = dataset[dataset['tor'].notna() & (dataset['tor'] != '')].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

pivot_table_tor['Total'] = pivot_table_tor.sum(axis=1)

# Add a "Total" row
pivot_table_tor.loc['Total'] = pivot_table_tor.sum(axis=0)

make_heatmap(pivot_table_tor, "heatmap_score_tor", fig_size=(20, 14))

In [ ]:
pivot_table_tor_bridge = dataset[dataset['tor_bridge'].notna() & (dataset['tor_bridge'] != '')].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

pivot_table_tor_bridge['Total'] = pivot_table_tor_bridge.sum(axis=1)

# Add a "Total" row
pivot_table_tor_bridge.loc['Total'] = pivot_table_tor_bridge.sum(axis=0)

make_heatmap(pivot_table_tor_bridge, "heatmap_score_tor_bridge", fig_size=(30,14))

In [ ]:
pivot_table_proxy = dataset[dataset['proxy'].notna() & (dataset['proxy'] != '')].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

pivot_table_proxy['Total'] = pivot_table_proxy.sum(axis=1)

# Add a "Total" row
pivot_table_proxy.loc['Total'] = pivot_table_proxy.sum(axis=0)

make_heatmap(pivot_table_proxy, "heatmap_score_proxy", take_last=True)

In [ ]:
pivot_table_proxy = dataset[dataset['i2p'].notna() & (dataset['i2p'] != '')].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

pivot_table_proxy['Total'] = pivot_table_proxy.sum(axis=1)

# Add a "Total" row
pivot_table_proxy.loc['Total'] = pivot_table_proxy.sum(axis=0)

make_heatmap(pivot_table_proxy, "heatmap_score_i2p")

In [ ]:
families = ["SINGLETON", "inmobi", "smsreg", "hiddad", "youmi", "kuguo", "bian"]

data_copy = dataset[dataset["family"].isin(families)].copy()

def get_category(row):
    categories = []
    if pd.notna(row['tor']):
        categories.append('Tor')
    if pd.notna(row['tor_bridge']):
        categories.append('TorPT')
    if pd.notna(row['vpn']):
        categories.append('VPN')
    if pd.notna(row['proxy']):
        categories.append('Proxy')
    if pd.notna(row['i2p']):
        categories.append('I2P')
    
    if not categories:
        return "none"
    
    return ', '.join(categories)

data_copy['category'] = data_copy.apply(get_category, axis=1)

pivot_table = data_copy.pivot_table(index='family', columns='category', aggfunc='size', fill_value=0)
pivot_table['Total'] = pivot_table.sum(axis=1)
pivot_table.loc['Total'] = pivot_table.sum(axis=0)

take_last = True

fig = plt.figure(figsize=(22,18))
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.01, 1])  # Allocate 10% height for the colorbar and the rest for the heatmap

# Create the axis for the colorbar
cbar_ax = fig.add_subplot(gs[0])  # Top axis for the colorbar

# Create the axis for the heatmap
heatmap_ax = fig.add_subplot(gs[-1])  # Bottom axis for the heatmap

# Plot the heatmap
heatmap = sns.heatmap(
    pivot_table,
    ax=heatmap_ax,
    cmap='Blues',           # Cyan color palette
    linewidths=0.5,         # Add spacing between boxes
    linecolor='lightgrey',  # Light grid color
    annot=True,             # Display counts within boxes
    fmt='g',                # Format annotations as integers
    annot_kws={"fontsize": 18, "rotation": 45, "fontweight": 'bold'},  # Rotate annotations diagonally and increase font size
    cbar_ax=cbar_ax,        # Use the custom colorbar axis
    cbar_kws={'orientation': 'horizontal'}  # Set colorbar orientation to horizontal
)

for spine in heatmap_ax.spines.values():
    spine.set_visible(True)      # Make the spines visible
    spine.set_color('black')     # Set border color
    spine.set_linewidth(2)       # Set border width

# Customize the horizontal colorbar
colorbar = heatmap.collections[0].colorbar  # Access the color bar
max_val = int(colorbar.vmax)  # Maximum value of the color bar
if not take_last:
    tick_positions = np.concatenate([np.arange(0, max_val + 10000, 10000)[:-1], [max_val]])  # Tick positions with 10K intervals
else:
    tick_positions = np.arange(0, max_val + 10000, 10000)[:-1]
colorbar.set_ticks(tick_positions)  # Apply ticks
if max_val > 50000:
    colorbar.set_ticklabels([f"{int(val / 1000)}K" for val in tick_positions], fontweight="bold", fontsize=10)  # Format tick labels

colorbar.set_label(
    "Counts for Statically Validated APKs", 
    fontsize=20, fontweight='bold', labelpad=10  # Add label to the colorbar
)

# Increase font size of color bar ticks
colorbar.ax.tick_params(labelsize=16)

# Customize heatmap axes
heatmap_ax.set_xticks(heatmap_ax.get_xticks())
heatmap_ax.set_xticklabels(heatmap_ax.get_xticklabels(), fontsize=24, fontweight='bold', rotation=90)
heatmap_ax.set_yticklabels(heatmap_ax.get_yticklabels(), fontsize=26, fontweight='bold', rotation=0)
heatmap_ax.set_xlabel("Covert Channel", fontsize=26, fontweight='bold')
heatmap_ax.set_ylabel("Malware Family", fontsize=30, fontweight='bold')

# Customize spines
for spine in heatmap_ax.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

# Save and show the plot
plt.savefig("<output directory>/heatmap_family.png", bbox_inches='tight')
plt.show()

In [ ]:
# Create the pivot table
# Filter out "Not a malware" and take top 25 by family count
top_families = dataset[dataset['family'] != 'Not a malware']['family'].value_counts().head(25).index
pivot_table = dataset[dataset['family'].isin(top_families)].pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

pivot_table['Total'] = pivot_table.sum(axis=1)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


# Create a figure and gridspec layout
fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[0.1, 0.02, 1])  # Allocate 10% height for the colorbar and the rest for the heatmap

# Create the axis for the colorbar
cbar_ax = fig.add_subplot(gs[0])  # Top axis for the colorbar

# Create the axis for the heatmap
heatmap_ax = fig.add_subplot(gs[-1])  # Bottom axis for the heatmap

# Plot the heatmap
heatmap = sns.heatmap(
    pivot_table,
    ax=heatmap_ax,
    cmap='Blues',           # Cyan color palette
    linewidths=0.5,         # Add spacing between boxes
    linecolor='lightgrey',  # Light grid color
    annot=True,             # Display counts within boxes
    fmt='g',                # Format annotations as integers
    annot_kws={"fontsize": 8, "rotation": 45},  # Rotate annotations diagonally and increase font size
    cbar_ax=cbar_ax,        # Use the custom colorbar axis
    cbar_kws={'orientation': 'horizontal'}  # Set colorbar orientation to horizontal
)

# Customize the horizontal colorbar
colorbar = heatmap.collections[0].colorbar  # Access the color bar
max_val = int(colorbar.vmax)  # Maximum value of the color bar
tick_positions = np.concatenate([np.arange(0, max_val + 10000, 10000)[:-1], [max_val]])   # Tick positions with 10K intervals
colorbar.set_ticks(tick_positions)  # Apply ticks
colorbar.set_ticklabels([f"{int(val / 1000)}K" for val in tick_positions])  # Format tick labels
colorbar.set_label(
    "Counts for Statically Validation APKs", 
    fontsize=20, fontweight='bold', labelpad=10  # Add label to the colorbar
)

# Increase font size of color bar ticks
colorbar.ax.tick_params(labelsize=16)

# Customize heatmap axes
heatmap_ax.set_xticks(heatmap_ax.get_xticks())
heatmap_ax.set_xticklabels(heatmap_ax.get_xticklabels(), fontsize=18, fontweight='bold', rotation=90)
heatmap_ax.set_yticklabels(heatmap_ax.get_yticklabels(), fontsize=18, fontweight='bold', rotation=0)
heatmap_ax.set_xlabel("Malware Family", fontsize=20, fontweight='bold')
heatmap_ax.set_ylabel("Year", fontsize=20, fontweight='bold')

# Customize spines
for spine in heatmap_ax.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

# Save and show the plot
plt.savefig("<output directory>/heatmap_family.png", bbox_inches='tight')
plt.show()


In [ ]:
# Create the pivot table
# Filter out "Not a malware" and take top 25 by family count
top_families = dataset[dataset['family'] != 'Not a malware']['family'].value_counts().head(50).index

pivot_table = dataset[dataset['family'].isin(top_families)].pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

pivot_table['Total'] = pivot_table.sum(axis=1)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


# Create a figure and gridspec layout
fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[0.1, 0.02, 1])  # Allocate 10% height for the colorbar and the rest for the heatmap

# Create the axis for the colorbar
cbar_ax = fig.add_subplot(gs[0])  # Top axis for the colorbar

# Create the axis for the heatmap
heatmap_ax = fig.add_subplot(gs[-1])  # Bottom axis for the heatmap

# Plot the heatmap
heatmap = sns.heatmap(
    pivot_table,
    ax=heatmap_ax,
    cmap='Blues',           # Cyan color palette
    linewidths=0.5,         # Add spacing between boxes
    linecolor='lightgrey',  # Light grid color
    annot=True,             # Display counts within boxes
    fmt='g',                # Format annotations as integers
    annot_kws={"fontsize": 8, "rotation": 45},  # Rotate annotations diagonally and increase font size
    cbar_ax=cbar_ax,        # Use the custom colorbar axis
    cbar_kws={'orientation': 'horizontal'}  # Set colorbar orientation to horizontal
)

# Customize the horizontal colorbar
colorbar = heatmap.collections[0].colorbar  # Access the color bar
max_val = int(colorbar.vmax)  # Maximum value of the color bar
tick_positions = np.concatenate([np.arange(0, max_val + 10000, 10000)[:-1], [max_val]])   # Tick positions with 10K intervals
colorbar.set_ticks(tick_positions)  # Apply ticks
colorbar.set_ticklabels([f"{int(val / 1000)}K" for val in tick_positions])  # Format tick labels
colorbar.set_label(
    "Counts for Statically Validation APKs", 
    fontsize=20, fontweight='bold', labelpad=10  # Add label to the colorbar
)

# Increase font size of color bar ticks
colorbar.ax.tick_params(labelsize=16)

# Customize heatmap axes
heatmap_ax.set_xticks(heatmap_ax.get_xticks())
heatmap_ax.set_xticklabels(heatmap_ax.get_xticklabels(), fontsize=18, fontweight='bold', rotation=90)
heatmap_ax.set_yticklabels(heatmap_ax.get_yticklabels(), fontsize=18, fontweight='bold', rotation=0)
heatmap_ax.set_xlabel("Malware Family", fontsize=20, fontweight='bold')
heatmap_ax.set_ylabel("Year", fontsize=20, fontweight='bold')

# Customize spines
for spine in heatmap_ax.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

# Save and show the plot
plt.savefig("<output directory>/heatmap_family.png", bbox_inches='tight')
plt.show()


In [ ]:
# Get the top 50 families by count
top_families = dataset[dataset['family'] != 'Not a malware']['family'].value_counts().head(25)

x_pos = np.arange(len(top_families)) * 0.2

# plt.figure(figsize=(20, 8))
# bars = plt.bar(
#     top_families.index, 
#     top_families.values, 
#     color='white', 
#     edgecolor='black', 
#     hatch="x",
#     width = 0.2
# )

plt.figure(figsize=(6, 4))
bars = plt.bar(
    x_pos,
    top_families.values,
    color='white',
    edgecolor='black',
    hatch='x',
    width=0.1  # keep width the same
)

# Add grid and customize
plt.grid(axis='both', linestyle='--', color='lightgrey', alpha=0.7)

# Set labels and format y-axis ticks
plt.xlabel('Malware Family', fontsize=10, fontweight='bold')
plt.ylabel('Count of Statically\nValidated APKs', fontsize=10, fontweight='bold')
# plt.xticks(rotation=90, fontsize=16, fontweight='bold')
plt.xticks(x_pos, top_families.index, rotation=90, fontsize=10, fontweight='bold')
plt.yticks(fontsize=10, fontweight='bold')

# Set y-axis to log scale
plt.yscale('log')

# Save plot
plt.tight_layout()
plt.savefig("<output directory>/family_counts.png")
plt.show()
plt.close()

In [ ]:
len(set(dataset[dataset["vpn"].notna()]["family"]) - set(["SINGLETON", "Not a malware"]))

In [ ]:
total_family_count = len(set(dataset["family"]) - set(["SINGLETON", "Not a malware"]))
tor_family_count = len(set(dataset[dataset["tor"].notna()]["family"]) - set(["SINGLETON", "Not a malware"]))
torpt_family_count = len(set(dataset[dataset["tor_bridge"].notna()]["family"]) - set(["SINGLETON", "Not a malware"]))
vpn_family_count = len(set(dataset[dataset["vpn"].notna()]["family"]) - set(["SINGLETON", "Not a malware"]))
proxy_family_count = len(set(dataset[dataset["proxy"].notna()]["family"]) - set(["SINGLETON", "Not a malware"]))
i2p_family_count = len(set(dataset[dataset["i2p"].notna()]["family"]) - set(["SINGLETON", "Not a malware"]))

# len(set(dataset[dataset["proxy"].notna()]["family"]) - set(["SINGLETON", "Not a malware"]))

print(f"Total Distinct families: {total_family_count}")
print(f"Distinct families with Tor: {tor_family_count}")
print(f"Distinct families with TorPT: {torpt_family_count}")
print(f"Distinct families with VPN: {vpn_family_count}")
print(f"Distinct families with Proxy: {proxy_family_count}")
print(f"Distinct families with I2P: {i2p_family_count}")

In [ ]:
from IPython.display import display

# make table of family vs all combinations of tor, tor_bridge, vpn, proxy

def get_combination(row):
    s = []
    if pd.notna(row['tor']):
        s.append('tor')
    if pd.notna(row['tor_bridge']):
        s.append('tor_bridge')
    if pd.notna(row['vpn']):
        s.append('vpn')
    if pd.notna(row['proxy']):
        s.append('proxy')
    if pd.notna(row['i2p']):
        s.append('i2p')

    return ", ".join(s)

# make table of family vs all combinations of tor, tor_bridge, vpn, proxy
dataset['combination'] = dataset.apply(get_combination, axis=1)
family_combination_table = dataset.pivot_table(index='family', columns='combination', aggfunc='size', fill_value=0)
# Reorder the columns to be in the desired order
desired_order = ['tor', 'tor_bridge', 'vpn', 'proxy', 'tor, tor_bridge', 'tor, vpn', 'tor, proxy', 'tor_bridge, vpn', 'tor_bridge, proxy', 'vpn, proxy', 'tor, tor_bridge, vpn', 'tor, tor_bridge, proxy', 'tor, vpn, proxy', 'tor_bridge, vpn, proxy']
family_combination_table = family_combination_table.reindex(columns=desired_order, fill_value=0)
# add total column
family_combination_table['Total'] = family_combination_table.sum(axis=1)
# sort by total
family_combination_table = family_combination_table.sort_values(by='Total', ascending=False)
display(family_combination_table.head(20))

In [ ]:
family_combination_table.to_csv("<output directory>/family_combination_table.csv")

In [ ]:
filtered_dataset = dataset.copy()

filtered_dataset["tor"] = filtered_dataset['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
filtered_dataset["vpn"] = filtered_dataset['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
filtered_dataset["tor_bridge"] = filtered_dataset['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)
filtered_dataset["proxy"] = filtered_dataset['proxy'].apply(lambda x: 1 if not pd.isna(x) else 0)
filtered_dataset["i2p"] = filtered_dataset['i2p'].apply(lambda x: 1 if not pd.isna(x) else 0)


chosen_families = ["SINGLETON", "inmobi", "smsreg", "hiddad", "youmi", "kuguo", "bian"]
chosen_families_data = filtered_dataset[dataset['family'].isin(chosen_families)].copy()

# categories = ["tor", "tor_bridge", "vpn", "proxy", 'i2p']
categories = ["tor", "i2p", "tor_bridge", "vpn", "proxy"]
aggregated_data = chosen_families_data.groupby("family")[categories].sum()

symbols = ['o', 's', '^', 'D', 'v', 'p', '*']

plt.figure(figsize=(8, 4.5))
for marker, family in zip(symbols, chosen_families):
    if family in aggregated_data.index:
        family_data = aggregated_data.loc[family]
        # Filter out zero values and their corresponding categories
        non_zero_mask = family_data > 0
        filtered_categories = [cat for cat, mask in zip(categories, non_zero_mask) if mask]
        filtered_values = family_data[non_zero_mask]

        # Plot only the non-zero points
        plt.plot(
            filtered_categories,
            filtered_values,
            label=family,
            linestyle='--',
            marker=marker,  # Add markers for better visibility
            markersize=10   # Increase marker size
        )


plt.xlabel('Covert Channel', fontsize=20)
plt.ylabel('Count', fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.xticks(categories, ["Tor", 'I2P', "TorPT", "VPN", "Proxy"], fontsize=18, fontweight="bold")  # Bold x-ticks

# yticks = plt.gca().get_yticks()  # Get current y-tick positions
# positive_yticks = [y for y in yticks if y >= 0]  # Filter out negative values
# plt.gca().set_yticks(positive_yticks)  # Set filtered ticks
# plt.gca().set_yticklabels([f"{int(y / 1000)}" for y in positive_yticks], fontsize=12, fontweight='bold')

plt.yscale('log')
yticks = plt.gca().get_yticks()  # Get current y-tick positions
for ticks in plt.gca().get_yticklabels():
    ticks.set_fontweight('bold')

# plt.legend(fontsize=14)
plt.grid()
plt.legend(fontsize=12, title="Families", title_fontsize=14, loc='upper left')  # Adjust legend size and title
plt.savefig("<path-to-directory>")
plt.show()


In [ ]:
filtered_dataset = dataset[
    ~dataset['family'].isin(['SINGLETON', 'Not a malware'])
].copy()

# Step 2: Calculate the total counts for each attribute in the filtered dataset
# total_proxy = filtered_dataset["proxy"].notna().sum()
# total_vpn = filtered_dataset["vpn"].notna().sum()
# total_tor = filtered_dataset["tor"].notna().sum()
# total_tor_bridge = filtered_dataset["tor_bridge"].notna().sum()

# Step 3: Filter for chosen families and encode presence of attributes as binary
chosen_families = ["SINGLETON", "inmobi", "smsreg", "hiddad", "youmi", "kuguo", "bian"]
chosen_families_data = dataset[dataset['family'].isin(chosen_families)].copy()

total_proxy = chosen_families_data["proxy"].notna().sum()
total_vpn = chosen_families_data["vpn"].notna().sum()
total_tor = chosen_families_data["tor"].notna().sum()
total_tor_bridge = chosen_families_data["tor_bridge"].notna().sum()

chosen_families_data['tor'] = chosen_families_data['tor'].notna().astype(int)
chosen_families_data['tor_bridge'] = chosen_families_data['tor_bridge'].notna().astype(int)
chosen_families_data['vpn'] = chosen_families_data['vpn'].notna().astype(int)
chosen_families_data['proxy'] = chosen_families_data['proxy'].notna().astype(int)

# Step 4: Aggregate data for radar chart
attributes = ['tor', 'tor_bridge', 'vpn', 'proxy']
labels = ["Tor", "TorPT", "VPN", "Proxy"]
family_counts = chosen_families_data.groupby('family')[attributes].sum()

# Step 5: Normalize counts by the total for each attribute in the filtered dataset
normalized_counts = family_counts.copy()
normalized_counts['tor'] = normalized_counts['tor'] / total_tor
normalized_counts['tor_bridge'] = normalized_counts['tor_bridge'] / total_tor_bridge
normalized_counts['vpn'] = normalized_counts['vpn'] / total_vpn
normalized_counts['proxy'] = normalized_counts['proxy'] / total_proxy

# Step 6: Plot the radar chart
num_vars = len(attributes)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Close the circle

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for idx, (family, row) in enumerate(normalized_counts.iterrows()):
    values = row.tolist()
    values += values[:1]  # Close the circle
    ax.plot(angles, values, label=family, linewidth=2)
    ax.fill(angles, values, alpha=0.25)

ax.set_ylim(0, 1)

# Add labels and title
ax.set_yticks(np.linspace(0, 1, 5))
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_title('Chosen Families - Radar Chart', size=15, position=(0.5, 1.1))
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.savefig("<path-to-directory>", bbox_inches='tight')
plt.show()

In [ ]:
# IGNORE THIS CELL

fig = plt.figure(figsize=(150, 10))
gs = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[0.1, 0.02, 1])  # Allocate 10% height for the colorbar and the rest for the heatmap

# Create the axis for the colorbar
cbar_ax = fig.add_subplot(gs[0])  # Top axis for the colorbar

# Create the axis for the heatmap
heatmap_ax = fig.add_subplot(gs[-1])  # Bottom axis for the heatmap

# Plot the heatmap
heatmap = sns.heatmap(
    pivot_table,
    ax=heatmap_ax,
    cmap='Blues',           # Cyan color palette
    linewidths=0.5,         # Add spacing between boxes
    linecolor='lightgrey',  # Light grid color
    annot=True,             # Display counts within boxes
    fmt='g',                # Format annotations as integers
    annot_kws={"fontsize": 8, "rotation": 45},  # Rotate annotations diagonally and increase font size
    cbar_ax=cbar_ax,        # Use the custom colorbar axis
    cbar_kws={'orientation': 'horizontal'}  # Set colorbar orientation to horizontal
)

# Customize the horizontal colorbar
colorbar = heatmap.collections[0].colorbar  # Access the color bar
max_val = int(colorbar.vmax)  # Maximum value of the color bar
tick_positions = np.concatenate([np.arange(0, max_val + 10000, 10000)[:-1], [max_val]])   # Tick positions with 10K intervals
colorbar.set_ticks(tick_positions)  # Apply ticks
colorbar.set_ticklabels([f"{int(val / 1000)}K" for val in tick_positions])  # Format tick labels
colorbar.set_label(
    "Counts for Statically Validation APKs", 
    fontsize=20, fontweight='bold', labelpad=10  # Add label to the colorbar
)

# Increase font size of color bar ticks
colorbar.ax.tick_params(labelsize=16)

# Customize heatmap axes
heatmap_ax.set_xticks(heatmap_ax.get_xticks())
heatmap_ax.set_xticklabels(heatmap_ax.get_xticklabels(), fontsize=18, fontweight='bold')
heatmap_ax.set_yticklabels(heatmap_ax.get_yticklabels(), fontsize=18, fontweight='bold', rotation=0)
heatmap_ax.set_xlabel("Malware Family", fontsize=20, fontweight='bold')
heatmap_ax.set_ylabel("Year", fontsize=20, fontweight='bold')

# Customize spines
for spine in heatmap_ax.spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

# Save and show the plot
plt.savefig("<path-to-directory>", bbox_inches='tight')
plt.show()

In [ ]:
dataset["family"].value_counts().sort_values(ascending=False).head(20)

In [ ]:
pivot_table_vpn = dataset[dataset['vpn'].notna() & (dataset['vpn'] != '')].pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

pivot_table_vpn['Total'] = pivot_table_vpn.sum(axis=1)

# Add a "Total" row
pivot_table_vpn.loc['Total'] = pivot_table_vpn.sum(axis=0)

# Plot the heatmap with totals
plt.figure(figsize=(100, 10))
sns.heatmap(
    pivot_table_vpn,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='g'          # Format annotations as integers
)

plt.title("Heatmap for family with Totals for VPN", fontsize=16)
plt.show()

In [ ]:
pivot_table_tor = dataset[dataset['tor'].notna() & (dataset['tor'] != '')].pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

pivot_table_tor['Total'] = pivot_table_tor.sum(axis=1)

# Add a "Total" row
pivot_table_tor.loc['Total'] = pivot_table_tor.sum(axis=0)

# Plot the heatmap with totals
plt.figure(figsize=(100, 10))
sns.heatmap(
    pivot_table_tor,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='g'          # Format annotations as integers
)

plt.title("Heatmap for family with Totals for TOR", fontsize=16)
plt.show()

In [ ]:
pivot_table_tor_bridge = dataset[dataset['tor_bridge'].notna() & (dataset['tor_bridge'] != '')].pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

pivot_table_tor_bridge['Total'] = pivot_table_tor_bridge.sum(axis=1)

# Add a "Total" row
pivot_table_tor_bridge.loc['Total'] = pivot_table_tor_bridge.sum(axis=0)

# Plot the heatmap with totals
plt.figure(figsize=(30, 10))
sns.heatmap(
    pivot_table_tor_bridge,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='g'          # Format annotations as integers
)

plt.title("Heatmap for family with Totals for TOR BRIDGE", fontsize=16)
plt.show()

In [ ]:
# Create the pivot table
pivot_table = dataset.pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

totals = pivot_table.sum(axis=1)

# Calculate percentages
pivot_table_percentage = pivot_table.div(totals, axis=0) * 100

# Plot the heatmap with percentages
plt.figure(figsize=(30, 10))
sns.heatmap(
    pivot_table_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',       # Format annotations as floats with 2 decimal places
    vmax=100         # Set maximum value for the heatmap
)

plt.title("Heatmap with Totals (Percentages)", fontsize=16)
plt.show()

In [ ]:
pivot_table_vpn = dataset[dataset['vpn'].notna() & (dataset['vpn'] != '')].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

totals_vpn = pivot_table_vpn.sum(axis=1)

# Calculate percentages
pivot_table_vpn_percentage = pivot_table_vpn.div(totals_vpn, axis=0) * 100

# Plot the heatmap with percentages
plt.figure(figsize=(30, 10))
sns.heatmap(
    pivot_table_vpn_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',        # Format annotations as floats with 2 decimal places
    vmax=100
)

plt.title("Heatmap with Totals for VPN (Percentages)", fontsize=16)
plt.show()

In [ ]:
# Create the pivot table
pivot_table_tor = dataset[dataset['tor'].notna() & (dataset['tor'] != '')].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

totals_tor = pivot_table_tor.sum(axis=1)

# Calculate percentages
pivot_table_tor_percentage = pivot_table_tor.div(totals_tor, axis=0) * 100

# Plot the heatmap with percentages
plt.figure(figsize=(30, 10))
sns.heatmap(
    pivot_table_tor_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',       # Format annotations as floats with 2 decimal places
    vmax=100         # Set maximum value for the heatmap
)

plt.title("Heatmap with Totals for TOR (Percentages)", fontsize=16)
plt.show()

In [ ]:
pivot_table_tor_bridge = dataset[dataset['tor_bridge'].notna() & (dataset['tor_bridge'] != '')].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

totals_tor_bridge = pivot_table_tor_bridge.sum(axis=1)

# Calculate percentages
pivot_table_tor_bridge_percentage = pivot_table_tor_bridge.div(totals_tor_bridge, axis=0) * 100

# Plot the heatmap with percentages
plt.figure(figsize=(30, 10))
sns.heatmap(
    pivot_table_tor_bridge_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',       # Format annotations as floats with 2 decimal places
    vmax=100         # Set maximum value for the heatmap
)

plt.title("Heatmap with Totals for TOR BRIDGE (Percentages)", fontsize=16)
plt.show()

In [ ]:
# Create the pivot table
pivot_table = dataset.pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

totals = pivot_table.sum(axis=1)

# Calculate percentages
pivot_table_percentage = pivot_table.div(totals, axis=0) * 100

# Plot the heatmap with percentages
plt.figure(figsize=(100, 10))
sns.heatmap(
    pivot_table_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',       # Format annotations as floats with 2 decimal places
    vmax=100         # Set maximum value for the heatmap
)

plt.title("Heatmap for family with Totals (Percentages)", fontsize=16)
plt.show()

In [ ]:
# Create the pivot table
pivot_table_vpn = dataset[dataset['vpn'].notna() & (dataset['vpn'] != '')].pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

totals_vpn = pivot_table_vpn.sum(axis=1)

# Calculate percentages
pivot_table_vpn_percentage = pivot_table_vpn.div(totals_vpn, axis=0) * 100

# Plot the heatmap with percentages
plt.figure(figsize=(100, 10))
sns.heatmap(
    pivot_table_vpn_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',       # Format annotations as floats with 2 decimal places
    vmax=100         # Set maximum value for the heatmap
)

plt.title("Heatmap for family with Totals for VPN (Percentages)", fontsize=16)
plt.show()

In [ ]:
pivot_table_tor = dataset[dataset['tor'].notna() & (dataset['tor'] != '')].pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

totals_tor = pivot_table_tor.sum(axis=1)

pivot_table_tor_percentage = pivot_table_tor.div(totals_tor, axis=0) * 100

# Plot the heatmap with totals
plt.figure(figsize=(100, 10))
sns.heatmap(
    pivot_table_tor_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',       # Format annotations as floats with 2 decimal places
    vmax=100         # Set maximum value for the heatmap
)

plt.title("Heatmap for family with Totals for TOR (Percentages)", fontsize=16)
plt.show()

In [ ]:
pivot_table_tor_bridge = dataset[dataset['tor_bridge'].notna() & (dataset['tor_bridge'] != '')].pivot_table(index='year', columns='family', aggfunc='size', fill_value=0)

totals_tor_bridge = pivot_table_tor_bridge.sum(axis=1)

pivot_table_tor_bridge_percentage = pivot_table_tor_bridge.div(totals_tor_bridge, axis=0) * 100

# Plot the heatmap with totals
plt.figure(figsize=(30, 10))
sns.heatmap(
    pivot_table_tor_bridge_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',       # Format annotations as floats with 2 decimal places
    vmax=100         # Set maximum value for the heatmap
)

plt.title("Heatmap for family with Totals for TOR BRIDGE (Percentages)", fontsize=16)
plt.show()

In [ ]:
# display table of family and count with percentage
family_count = dataset['family'].value_counts()
family_count["Total"] = family_count.sum()
family_count = family_count.apply(lambda x: f"{x} ({(x/family_count['Total'])*100:.2f}%)")
print(family_count.to_latex())

In [ ]:
from upsetplot import UpSet

new_data = dataset.copy()

new_data["tor"] = new_data['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
new_data["vpn"] = new_data['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
new_data["tor_bridge"] = new_data['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)
new_data["proxy"] = new_data['proxy'].apply(lambda x: 1 if not pd.isna(x) else 0)
new_data["i2p"] = new_data['i2p'].apply(lambda x: 1 if not pd.isna(x) else 0)

new_data = new_data.groupby(['tor', 'vpn', 'tor_bridge', "proxy", "i2p"]).size()

UpSet(new_data).plot()

plt.show()

In [ ]:
# set month = 1 where month = 0

dataset['month'] = dataset['month'].apply(lambda x: 1 if x == 0 else x)
dataset['date'] = pd.to_datetime(dataset[['year', 'month']].assign(day=1))

In [ ]:
counts_per_month = dataset.groupby('year').size()
cumulative_counts = counts_per_month.cumsum()

plt.figure(figsize=(8, 4.5))
plt.bar(cumulative_counts.index, cumulative_counts, color='white', 
        edgecolor='black', 
        hatch="*")
plt.plot(cumulative_counts, marker='o', linestyle='--', color='r', label='Cumulative Count')
# plt.title('Cumulative Counts Over Time', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Cumulative Count (in k)', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.xticks(cumulative_counts.index, fontsize=12, fontweight="bold", rotation=45)

yticks = plt.gca().get_yticks()  # Get current y-tick positions
positive_yticks = [y for y in yticks if y >= 0]  # Filter out negative values
plt.gca().set_yticks(positive_yticks)  # Set filtered ticks
plt.gca().set_yticklabels([f"{int(y / 1000)}" for y in positive_yticks], fontsize=12, fontweight='bold')

# plt.legend(fontsize=14)
plt.grid()
plt.savefig("<path-to-directory>")
plt.show()


In [ ]:
# plot the percentages of samples per year in a line graph
percentages = {}
for year in year_totals_malware:
    if year_totals_malware[year] != 0:
        percentages[year] = dataset[dataset['year'] == year].shape[0] / year_totals_malware[year] * 100

plt.figure(figsize=(20, 10))
plt.plot(list(percentages.keys()), list(percentages.values()), marker='o', linestyle='-', color='b')
plt.title('Percentages of Samples per Year', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Percentage of Samples', fontsize=14)
plt.grid()
plt.show()


In [ ]:
vpn_counts = dataset[dataset['vpn'].notna()].groupby('year')['vpn'].count()
tor_counts = dataset[dataset['tor'].notna()].groupby('year')['tor'].count()
tor_bridge_counts = dataset[dataset['tor_bridge'].notna()].groupby('year')['tor_bridge'].count()
proxy_counts = dataset[dataset['proxy'].notna()].groupby('year')['proxy'].count()
i2p_counts = dataset[dataset['i2p'].notna()].groupby('year')['i2p'].count()

# Get all unique years in the dataset
all_years = sorted(dataset['year'].unique())

# Create subplots
from matplotlib.ticker import FuncFormatter

# Define a function to format large y-tick values
def format_large_ticks(value, _):
    if value >= 5000:
        return f'{value / 1000:.0f}k'
    return f'{int(value)}'

# Function to create and save individual plots
def create_plot(data, years, file_name, pattern):
    plt.figure(figsize=(4.5,3))
    bars = plt.bar(
        years, 
        data.reindex(years, fill_value=0), 
        color='white', 
        edgecolor='black', 
        hatch=pattern,
        width=0.6
    )
    
    # Add grid and customize
    plt.grid(axis='both', linestyle='--', color='lightgrey', alpha=0.7)
    
    # Set labels and format y-axis ticks
    plt.xlabel('Year', fontsize=14, fontweight='bold')
    if file_name == "tor_counts":
        plt.ylabel('Count of Statically\nValidated APKs', fontsize=14, fontweight='bold')
    plt.xticks(years, rotation=90, fontsize=14, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.gca().yaxis.set_major_formatter(FuncFormatter(format_large_ticks))
    
    # Save plot
    plt.tight_layout()
    plt.savefig(f"<path-to-directory>")
    plt.show()
    plt.close()

# Get all unique years in the dataset
all_years = sorted(dataset['year'].unique())

# Create and save individual plots
create_plot(vpn_counts, all_years, "vpn_counts", pattern="//")  # Diagonal lines
create_plot(tor_counts, all_years, "tor_counts", pattern="\\")  # Backward diagonal lines
create_plot(tor_bridge_counts, all_years, "tor_bridge_counts", pattern="..")  # Dots
create_plot(proxy_counts, all_years, "proxy_counts", pattern="++")  # Crosses
create_plot(i2p_counts, all_years, "i2p_counts", pattern="xx")  # Crosses

In [ ]:
# family count pi chart
family_count = dataset['family'].value_counts()
# remove SINGLETON and Not a malware from the pie chart
family_count = family_count[family_count.index != 'SINGLETON']
family_count = family_count[family_count.index != 'Not a malware']

plt.figure(figsize=(20, 10))
plt.pie(family_count[:-2], labels=family_count[:-2].index, autopct='%1.1f%%')
plt.title('Family Count', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
dataset.info()

In [ ]:
dataset.describe()

In [ ]:
dataset_times = pd.read_csv("<path-to-directory>")

dataset_times['first_seen'] = pd.to_datetime(dataset_times['first_seen'], unit='s')
dataset_times['last_seen'] = pd.to_datetime(dataset_times['last_seen'], unit='s')

dataset_times['time_range'] = dataset_times.apply(
    lambda row: pd.date_range(start=row['first_seen'], end=row['last_seen'], freq='M'), axis=1
)

expanded_data = dataset_times.explode('time_range')

# Step 4: Group by time period and count the number of active samples
expanded_data['time_range'] = expanded_data['time_range'].dt.to_period('M')  # Group by monthly periods
monthly_counts = expanded_data.groupby('time_range').size()

# Step 5: Compute cumulative counts
cumulative_counts = monthly_counts.cumsum()

# Step 6: Plot horizontal area chart with lines stopping at `last_seen`
plt.figure(figsize=(12, 6))
plt.fill_between(cumulative_counts.index.to_timestamp(), 
                 cumulative_counts.values, 
                 color="#FFA07A", alpha=0.8, step='pre')
plt.gca().invert_yaxis()  # Flip Y-axis to create a horizontal effect
plt.title('Cumulative Malware Samples Over Time (Lines Stop at Last Seen)')
plt.xlabel('Time')
plt.ylabel('Malware Samples')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
dir(dataset_times["time_range"][0])

In [ ]:
sum(dataset["size"].dropna()) / 1024 / 1024 / 1024

In [ ]:
dataset[~(dataset["tor"].notna() | dataset["tor_bridge"].notna() | dataset["proxy"].notna() | dataset["vpn"].notna() | dataset["i2p"].notna())]

In [ ]:
from upsetplot import UpSet

new_data = dataset.copy()

new_data["Tor"] = new_data['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
new_data["VPN"] = new_data['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
new_data["TorPT"] = new_data['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)
new_data["Proxy"] = new_data['proxy'].apply(lambda x: 1 if not pd.isna(x) else 0)
new_data["I2P"] = new_data['i2p'].apply(lambda x: 1 if not pd.isna(x) else 0)

new_data = new_data.groupby(['Tor', 'VPN', 'TorPT', 'Proxy', 'I2P']).size()

upset = UpSet(new_data)
axes = upset.plot()

axes['intersections'].set_yscale('log')

# Annotate bars with counts
for bar in axes['intersections'].patches:
    height = bar.get_height()
    if height > 0:  # Avoid annotating bars with height 0
        # bar.set_x(bar.get_x() + 0.2)  # Move the bar to the right by 0.2
        axes['intersections'].text(bar.get_x() + bar.get_width() / 2, 
                                   height, 
                                   f'{int(height)}', 
                                   ha='center', 
                                   va='bottom', 
                                   fontsize=8, 
                                   color='black')   
        
for bar in axes['totals'].patches:
    height = bar.get_width()  # Bar width corresponds to the count in totals

    if height > 0:  # Avoid annotating bars with width 0
        axes['totals'].text(height , 
                            bar.get_y() + bar.get_height() / 2, 
                            f'{int(height)}', 
                            ha='right', 
                            va='center', 
                            fontsize=7, 
                            color='black',
                            rotation=90)
        
axes["intersections"].tick_params(axis='both', which='major', labelsize=12)
# axes["totals"].tick_params(axis='both', which='major', labelsize=12)

axes["intersections"].set_ylabel("Number of\nStatically\nValidated APKs", fontsize=15, fontweight='bold')
axes["matrix"].set_xticks(axes["matrix"].get_xticks() + 1)

# # Adjust subplot spacing manually
# inter_pos = axes['intersections'].get_position()
# axes['intersections'].set_position([inter_pos.x0 - 0.06, inter_pos.y0, inter_pos.width + 0.06, inter_pos.height])

# # Optional: Adjust 'totals' and 'matrix' if needed
# totals_pos = axes['totals'].get_position()
# axes['totals'].set_position([totals_pos.x0 - 0.03, totals_pos.y0, totals_pos.width + 0.03, totals_pos.height])

# matrix_pos = axes['matrix'].get_position()
# axes['matrix'].set_position([matrix_pos.x0 - 0.06, matrix_pos.y0, matrix_pos.width + 0.06, matrix_pos.height])

# plt.subplots_adjust(left=0.08)

num_bars = len(axes['intersections'].patches)
axes['intersections'].set_xlim(-1, num_bars + 0.5)

# plt.tight_layout()
# plt.tight_layout()
plt.tight_layout()
plt.savefig("<path-to-directory>", bbox_inches='tight')

plt.show()

In [ ]:
# Create a copy of the dataset to avoid modifying the original
data_copy = dataset.copy()

# Replace NaN values with 0 and non-NaN values with 1 for vpn, tor, and tor_bridge
data_copy['vpn'] = data_copy['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor'] = data_copy['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor_bridge'] = data_copy['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)

# Sort the dataset by score in descending order
data_copy = data_copy.sort_values(by='score', ascending=False)

# Calculate cumulative counts for vpn, tor, and tor_bridge
data_copy['cumulative_vpn'] = data_copy['vpn'].cumsum()
data_copy['cumulative_tor'] = data_copy['tor'].cumsum()
data_copy['cumulative_tor_bridge'] = data_copy['tor_bridge'].cumsum()

# Calculate the total counts for vpn, tor, and tor_bridge
total_vpn = data_copy['vpn'].sum()
total_tor = data_copy['tor'].sum()
total_tor_bridge = data_copy['tor_bridge'].sum()

# Calculate cumulative percentages
data_copy['cumulative_vpn_pct'] = (data_copy['cumulative_vpn'] / total_vpn) * 100
data_copy['cumulative_tor_pct'] = (data_copy['cumulative_tor'] / total_tor) * 100
data_copy['cumulative_tor_bridge_pct'] = (data_copy['cumulative_tor_bridge'] / total_tor_bridge) * 100

# Plot the cumulative percentages
plt.figure(figsize=(20, 10))
plt.plot(data_copy['score'], data_copy['cumulative_vpn_pct'], label='VPN', color='b')
plt.plot(data_copy['score'], data_copy['cumulative_tor_pct'], label='TOR', color='g')
plt.plot(data_copy['score'], data_copy['cumulative_tor_bridge_pct'], label='TOR Bridge', color='r')

plt.title('Cumulative Percentage on Decreasing Score', fontsize=16)
plt.xlabel('Score', fontsize=14)
plt.ylabel('Cumulative Percentage', fontsize=14)
plt.legend()
plt.grid()
# plt.gca().invert_xaxis()  # Invert the x-axis
plt.show()


In [ ]:
pivot_table_singleton = dataset[dataset['family'] == "SINGLETON"].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

pivot_table_singleton['Total'] = pivot_table_singleton.sum(axis=1)

# Add a "Total" row
pivot_table_singleton.loc['Total'] = pivot_table_singleton.sum(axis=0)

# Plot the heatmap with totals
plt.figure(figsize=(30, 10))
sns.heatmap(
    pivot_table_singleton,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='g'          # Format annotations as integers
)

plt.title("Heatmap with Totals for SINGLETON", fontsize=16)
plt.show()

In [ ]:
pivot_table_singleton = dataset[dataset['family'] == "SINGLETON"].pivot_table(index='year', columns='score', aggfunc='size', fill_value=0)

total_singleton = pivot_table_singleton.sum(axis=1)

# Calculate percentages
pivot_table_singleton_percentage = pivot_table_singleton.div(total_singleton, axis=0) * 100

# Plot the heatmap with totals
plt.figure(figsize=(30, 10))
sns.heatmap(
    pivot_table_singleton_percentage,
    cmap='Blues',  # Cyan color palette
    linewidths=0.5,  # Add spacing between boxes
    annot=True,      # Display counts within boxes
    fmt='.2f',       # Format annotations as floats with 2 decimal places
    vmax=100         # Set maximum value for the heatmap
)

plt.title("Heatmap with Totals for SINGLETON (Percentages)", fontsize=16)
plt.show()

In [ ]:
import plotly.graph_objects as go

data_copy = dataset.copy()

# Replace NaN values with 0 and non-NaN values with 1 for vpn, tor, and tor_bridge
data_copy['vpn'] = data_copy['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor'] = data_copy['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor_bridge'] = data_copy['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['proxy'] = data_copy['proxy'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['i2p'] = data_copy['i2p'].apply(lambda x: 1 if not pd.isna(x) else 0)

# set data_copy to have only columns, package, year, tor, tor_bridge, vpn, proxy
# data_copy = data_copy[['package', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
data_copy = data_copy[['family', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
# data_copy = data_copy[data_copy['year'] >= 2019]

def get_category(row):
    categories = []
    if row['tor'] == 1:
        categories.append('Tor')
    if row['tor_bridge'] == 1:
        categories.append('TorPT')
    if row['vpn'] == 1:
        categories.append('VPN')
    if row['proxy'] == 1:
        categories.append('Proxy')
    if row['i2p'] == 1:
        categories.append('I2P')
    
    if not categories:
        return "none"
    
    return ', '.join(categories)

data_copy['category'] = data_copy.apply(get_category, axis=1)

# Find unique years and categories
years = sorted(data_copy['year'].unique())
categories = data_copy['category'].unique()

year_category_map = {}

color_palette = [
    "rgba(255, 182, 193, 0.7)",  # Light Pink
    "rgba(173, 216, 230, 0.7)",  # Light Blue
    "rgba(144, 238, 144, 0.7)",  # Light Green
    "rgba(255, 218, 185, 0.7)",  # Peach Puff
    "rgba(216, 191, 216, 0.7)",  # Thistle
    "rgba(255, 192, 203, 0.7)",  # Pink
    "rgba(222, 184, 135, 0.7)",  # Burlywood
    "rgba(176, 224, 230, 0.7)",  # Powder Blue
    "rgba(255, 160, 122, 0.7)",  # Light Salmon
    "rgba(255, 250, 205, 0.7)",  # Lemon Chiffon
    "rgba(240, 128, 128, 0.7)",  # Light Coral
    "rgba(135, 206, 250, 0.7)",  # Light Sky Blue
    "rgba(152, 251, 152, 0.7)",  # Pale Green
    "rgba(255, 228, 181, 0.7)",  # Moccasin
    "rgba(221, 160, 221, 0.7)"   # Plum
]
category_color = {category: color for category, color in zip(categories, color_palette)}

# Populate the dictionary with unique year-category combinations
for year in years:
    for category in categories:
        key = f"{year} - {category}"
        year_category_map[key] = len(year_category_map)

labels = list(year_category_map.keys())

node_colors = []
for label in labels:
    _, category = label.split(" - ")
    node_colors.append(category_color[category])

# Prepare Sankey data
sources = []
targets = []
values = []
link_colors = []


# packages = data_copy["package"].unique()
families = data_copy["family"].unique()


start_year = data_copy['year'].min()
end_year = data_copy['year'].max()

# for year in years[:-1]:
#     for category in categories:
#         key = f"{year} - {category}"
#         next_key = f"{year + 1} - {category}"

#         sources.append(year_category_map[key])
#         targets.append(year_category_map[next_key])
#         values.append(1)
#         link_colors.append("rgba(255, 255, 255, 1)")


# Mann modif start
# for package in packages:
#     package_data = data_copy[data_copy['package'] == package]

#     cur_year = package_data['year'].min()

#     package_years = list(sorted(package_data['year'].unique()))

#     for year in package_years[1:]:
#         next_year = year
#         cur_category = package_data[package_data['year'] == cur_year].iloc[0]['category']
#         next_category = package_data[package_data['year'] == next_year].iloc[0]['category']

#         while (cur_year < year-1):
#             source_label = f"{cur_year} - {cur_category}"
#             target_label = f"{cur_year + 1} - {cur_category}"

#             sources.append(year_category_map[source_label])
#             targets.append(year_category_map[target_label])
#             values.append(1)
#             link_colors.append(category_color[cur_category])

#             cur_year += 1

#         source_label = f"{cur_year} - {cur_category}"
#         target_label = f"{next_year} - {next_category}"

#         sources.append(year_category_map[source_label])
#         targets.append(year_category_map[target_label])
#         link_colors.append(category_color[cur_category])
#         values.append(1)

#         if cur_category != next_category:
#             print(f"MIL GAYA: {package}, {cur_year}, {next_year}, |{cur_category}| |{next_category}|")

#         cur_year = year

for family in families:
    family_data = data_copy[data_copy['family'] == family]

    cur_year = family_data['year'].min()
    family_years = list(sorted(family_data['year'].unique()))

    for year in family_years[1:]:
        next_year = year
        cur_category = family_data[family_data['year'] == cur_year].iloc[0]['category']
        next_category = family_data[family_data['year'] == next_year].iloc[0]['category']

        while (cur_year < year-1):
            source_label = f"{cur_year} - {cur_category}"
            target_label = f"{cur_year + 1} - {cur_category}"

            sources.append(year_category_map[source_label])
            targets.append(year_category_map[target_label])
            values.append(1)
            link_colors.append(category_color[cur_category])

            cur_year += 1

        source_label = f"{cur_year} - {cur_category}"
        target_label = f"{next_year} - {next_category}"

        sources.append(year_category_map[source_label])
        targets.append(year_category_map[target_label])
        link_colors.append(category_color[cur_category])
        values.append(1)

        if cur_category != next_category:
            print(f"MIL GAYA: {family}, {cur_year}, {next_year}, |{cur_category}| |{next_category}|")

        cur_year = year


    # Mann modif end


    # if year != years[-1]:
    #     row = package_data[package_data['year'] == cur_year].iloc[0]
    #     while (cur_year != years[-1]):
    #         next_year = cur_year + 1
    #         cur_category = row['category']
    #         next_category = row['category']

    #         source_label = f"{cur_year} - {cur_category}"
    #         target_label = f"{next_year} - {next_category}"

    #         sources.append(year_category_map[source_label])
    #         targets.append(year_category_map[target_label])
    #         values.append(1)
    #         link_colors.append(category_color[cur_category])

    #         cur_year += 1

# Create Sankey diagram

categories = set([label.split(" - ")[1] for label in labels])
categories = list(categories)

# Define the custom order
custom_order = ["Tor", "TorPT", "VPN", "Proxy", "I2P"]

# Helper function to get the custom order index
def get_custom_order_index(element):
    # Split by comma and find the index of each element in the custom order
    return [custom_order.index(e.strip()) for e in element.split(', ')]

# Sort by count of elements first (number of commas + 1), then by the custom order
sorted_categories = sorted(categories, key=lambda x: (len(x.split(', ')), get_custom_order_index(x)))

years = set([int(label.split(" - ")[0]) for label in labels])

bar_size_years = {}

for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    if year not in bar_size_years:
        bar_size_years[year] = {}

    if category not in bar_size_years[year]:
        idx = labels.index(label)
        bar_size_years[year][category] = max(sources.count(idx), targets.count(idx))

for year in bar_size_years:
    total = sum(bar_size_years[year].values())
    if total != 0:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = bar_size_years[year][category] / total
    else:
        for category in categories:
            if category in bar_size_years:
                bar_size_years[year].pop(category)

    diff = 0
    for category in sorted_categories:
        if category in bar_size_years[year]:
            temp = bar_size_years[year][category]
            bar_size_years[year][category] = diff
            diff += temp

    max_cat = max(bar_size_years[year].values())
    min_cat = min(bar_size_years[year].values())

    start = 0.02
    end = 0.98

    if max_cat != min_cat:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (bar_size_years[year][category] - min_cat) * (end - start) / (max_cat - min_cat)
    elif len(categories) >= 1:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (end - start) / len(categories) * sorted_categories.index(category)


y_positions = []
for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    y_positions.append(bar_size_years[year][category])


x_positions = [0.01 + (int(label.split(" - ")[0]) - min(years)) * (0.98) / (max(years) - min(years)) for label in labels]

to_remove = set(list(range(len(labels))))
to_remove -= set(sources)
to_remove -= set(targets)

change_map = {}
diff = 0
for idx in range(len(labels)):
    if idx in to_remove:
        # change_map[idx] = idx - diff
        labels.pop(idx - diff)
        x_positions.pop(idx - diff)
        y_positions.pop(idx - diff)
        node_colors.pop(idx - diff)
        diff += 1
    else:
        change_map[idx] = idx - diff

for idx in range(len(sources)):
    sources[idx] = change_map[sources[idx]]
    targets[idx] = change_map[targets[idx]]

fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        x = x_positions,
        y = y_positions,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Sankey Diagram of VPN, TOR, TorPT, Proxy Usage by Year",
    font_size=18,
    height=1200,
    width=6000
    
)

fig.write_json("<path-to-directory>")
fig.write_image("<path-to-directory>")
fig.write_html("<path-to-directory>")

fig.show()

In [ ]:
set(dataset["user"])

In [ ]:
"hi"

In [ ]:
pip install -U kaleido

In [ ]:
pip install -U plotly

# Extracting data for autoins

In [ ]:
print(dataset['year'].dtype)

In [ ]:
import plotly.graph_objects as go

data_copy = dataset.copy()


# desired_families = ['autoins', 'smsreg', 'hiddad', 'smssend', 'umpay', 'triada', 'ewind', 'kreditspy', 'dianjin', 'traca', 'smsspy', 'dataeye', 'cryxos']
# # Filter to only 'autoins' family
# data_copy = data_copy[data_copy['family'].isin(desired_families)]


# dropping all singleton
data_copy = data_copy[data_copy['family'] == 'SINGLETON']



# Replace NaN values with 0 and non-NaN values with 1 for vpn, tor, and tor_bridge
data_copy['vpn'] = data_copy['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor'] = data_copy['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor_bridge'] = data_copy['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['proxy'] = data_copy['proxy'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['i2p'] = data_copy['i2p'].apply(lambda x: 1 if not pd.isna(x) else 0)

# set data_copy to have only columns, package, year, tor, tor_bridge, vpn, proxy
# data_copy = data_copy[['package', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
data_copy = data_copy[['family', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
# data_copy = data_copy[['hash', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]

# print(data_copy.iloc[0])

data_copy = data_copy[data_copy['year'] >= 2019]

def get_category(row):
    categories = []
    if row['tor'] == 1:
        categories.append('Tor')
    if row['tor_bridge'] == 1:
        categories.append('TorPT')
    if row['vpn'] == 1:
        categories.append('VPN')
    if row['proxy'] == 1:
        categories.append('Proxy')
    if row['i2p'] == 1:
        categories.append('I2P')
    
    if not categories:
        return "none"
    
    return ', '.join(categories)

data_copy['category'] = data_copy.apply(get_category, axis=1)

# Find unique years and categories
years = sorted(data_copy['year'].unique())
categories = data_copy['category'].unique()

year_category_map = {}

color_palette = [
    "rgba(255, 182, 193, 0.7)",  # Light Pink
    "rgba(173, 216, 230, 0.7)",  # Light Blue
    "rgba(144, 238, 144, 0.7)",  # Light Green
    "rgba(255, 218, 185, 0.7)",  # Peach Puff
    "rgba(216, 191, 216, 0.7)",  # Thistle
    "rgba(255, 192, 203, 0.7)",  # Pink
    "rgba(222, 184, 135, 0.7)",  # Burlywood
    "rgba(176, 224, 230, 0.7)",  # Powder Blue
    "rgba(255, 160, 122, 0.7)",  # Light Salmon
    "rgba(255, 250, 205, 0.7)",  # Lemon Chiffon
    "rgba(240, 128, 128, 0.7)",  # Light Coral
    "rgba(135, 206, 250, 0.7)",  # Light Sky Blue
    "rgba(152, 251, 152, 0.7)",  # Pale Green
    "rgba(255, 228, 181, 0.7)",  # Moccasin
    "rgba(221, 160, 221, 0.7)",  # Plum
    "rgba(255, 105, 180, 0.7)",  # Hot Pink
    "rgba(100, 149, 237, 0.7)",  # Cornflower Blue
    "rgba(60, 179, 113, 0.7)",   # Medium Sea Green
    "rgba(255, 222, 173, 0.7)",  # Navajo White
    "rgba(238, 130, 238, 0.7)",  # Violet
    "rgba(255, 69, 0, 0.7)",     # Red Orange
    "rgba(0, 191, 255, 0.7)",    # Deep Sky Blue
    "rgba(34, 139, 34, 0.7)",    # Forest Green
    "rgba(255, 239, 213, 0.7)",  # Papaya Whip
    "rgba(186, 85, 211, 0.7)",   # Medium Orchid
    "rgba(255, 140, 0, 0.7)",    # Dark Orange
    "rgba(70, 130, 180, 0.7)",   # Steel Blue
    "rgba(46, 139, 87, 0.7)",    # Sea Green
    "rgba(255, 248, 220, 0.7)",  # Cornsilk
    "rgba(218, 112, 214, 0.7)",  # Orchid
    "rgba(255, 165, 0, 0.7)"     # Orange
]
category_color = {category: color for category, color in zip(categories, color_palette)}

# Populate the dictionary with unique year-category combinations
for year in years:
    for category in categories:
        key = f"{year} - {category}"
        year_category_map[key] = len(year_category_map)

labels = list(year_category_map.keys())

node_colors = []
for label in labels:
    _, category = label.split(" - ")
    node_colors.append(category_color[category])

# Prepare Sankey data
sources = []
targets = []
values = []
link_colors = []


# packages = data_copy["package"].unique()
families = data_copy["family"].unique()
# hashes = data_copy["hash"].unique()

start_year = data_copy['year'].min()
end_year = data_copy['year'].max()


from collections import Counter

# for family in families:
#     family_data = data_copy[data_copy['family'] == family]
#     family_years = sorted(family_data['year'].unique())

#     for i in range(len(family_years) - 1):
#         y1 = family_years[i]
#         y2 = family_years[i + 1]

#         data_y1 = family_data[family_data['year'] == y1]
#         data_y2 = family_data[family_data['year'] == y2]

#         # Count transitions between all categories in y1 and y2
#         for _, row1 in data_y1.iterrows():
#             cat1 = row1['category']

#             # Find the closest match in y2 (same hash/package logic not known, so assuming all-to-all mapping)
#             for _, row2 in data_y2.iterrows():
#                 cat2 = row2['category']

#                 # Handle in-between years
#                 for y_gap in range(y1, y2 - 1):
#                     intermediate_label = f"{y_gap} - {cat1}"
#                     next_label = f"{y_gap + 1} - {cat1}"
#                     sources.append(year_category_map[intermediate_label])
#                     targets.append(year_category_map[next_label])
#                     values.append(1)
#                     link_colors.append(category_color[cat1])

#                 # Final edge to next year
#                 source_label = f"{y1} - {cat1}"
#                 target_label = f"{y2} - {cat2}"
#                 sources.append(year_category_map[source_label])
#                 targets.append(year_category_map[target_label])
#                 values.append(1)
#                 link_colors.append(category_color[cat1])

#                 if cat1 != cat2:
#                     print(f"Transition: {family} from {y1} to {y2}: {cat1} → {cat2}")


seen_transitions_by_family = {}

for family in families:
    family_data = data_copy[data_copy['family'] == family]
    family_years = sorted(family_data['year'].unique())
    seen_transitions = set()

    for i in range(len(family_years) - 1):
        y1 = family_years[i]
        y2 = family_years[i + 1]

        data_y1 = family_data[family_data['year'] == y1]
        data_y2 = family_data[family_data['year'] == y2]

        for _, row1 in data_y1.iterrows():
            cat1 = row1['category']

            for _, row2 in data_y2.iterrows():
                cat2 = row2['category']

                # Handle in-between years
                for y_gap in range(y1, y2 - 1):
                    intermediate = (y_gap, cat1, y_gap + 1, cat1)
                    if intermediate not in seen_transitions:
                        src_label = f"{y_gap} - {cat1}"
                        tgt_label = f"{y_gap + 1} - {cat1}"

                        sources.append(year_category_map[src_label])
                        targets.append(year_category_map[tgt_label])
                        values.append(1)
                        link_colors.append(category_color[cat1])
                        seen_transitions.add(intermediate)

                final_transition = (y1, cat1, y2, cat2)
                if final_transition not in seen_transitions:
                    src_label = f"{y1} - {cat1}"
                    tgt_label = f"{y2} - {cat2}"

                    sources.append(year_category_map[src_label])
                    targets.append(year_category_map[tgt_label])
                    values.append(1)
                    link_colors.append(category_color[cat1])
                    seen_transitions.add(final_transition)

                    if cat1 != cat2:
                        print(f"Transition in {family}: {y1} {cat1} → {y2} {cat2}")

    seen_transitions_by_family[family] = seen_transitions

categories = set([label.split(" - ")[1] for label in labels])
categories = list(categories)

# Define the custom order
custom_order = ["Tor", "TorPT", "VPN", "Proxy", "I2P"]

# Helper function to get the custom order index
def get_custom_order_index(element):
    # Split by comma and find the index of each element in the custom order
    return [custom_order.index(e.strip()) for e in element.split(', ')]

# Sort by count of elements first (number of commas + 1), then by the custom order
sorted_categories = sorted(categories, key=lambda x: (len(x.split(', ')), get_custom_order_index(x)))

years = set([int(label.split(" - ")[0]) for label in labels])

bar_size_years = {}

for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    if year not in bar_size_years:
        bar_size_years[year] = {}

    if category not in bar_size_years[year]:
        idx = labels.index(label)
        bar_size_years[year][category] = max(sources.count(idx), targets.count(idx))

for year in bar_size_years:
    total = sum(bar_size_years[year].values())
    if total != 0:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = bar_size_years[year][category] / total
    else:
        for category in categories:
            if category in bar_size_years:
                bar_size_years[year].pop(category)

    diff = 0
    for category in sorted_categories:
        if category in bar_size_years[year]:
            temp = bar_size_years[year][category]
            bar_size_years[year][category] = diff
            diff += temp

    max_cat = max(bar_size_years[year].values())
    min_cat = min(bar_size_years[year].values())

    start = 0.02
    end = 0.98

    if max_cat != min_cat:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (bar_size_years[year][category] - min_cat) * (end - start) / (max_cat - min_cat)
    elif len(categories) >= 1:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (end - start) / len(categories) * sorted_categories.index(category)


y_positions = []
for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    y_positions.append(bar_size_years[year][category])


x_positions = [0.01 + (int(label.split(" - ")[0]) - min(years)) * (0.98) / (max(years) - min(years)) for label in labels]

to_remove = set(list(range(len(labels))))
to_remove -= set(sources)
to_remove -= set(targets)

change_map = {}
diff = 0
for idx in range(len(labels)):
    if idx in to_remove:
        # change_map[idx] = idx - diff
        labels.pop(idx - diff)
        x_positions.pop(idx - diff)
        y_positions.pop(idx - diff)
        node_colors.pop(idx - diff)
        diff += 1
    else:
        change_map[idx] = idx - diff

for idx in range(len(sources)):
    sources[idx] = change_map[sources[idx]]
    targets[idx] = change_map[targets[idx]]

fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        x = x_positions,
        y = y_positions,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Sankey Diagram of VPN, TOR, TorPT, Proxy Usage by Year",
    font_size=18,
    height=1200,
    width=6000
    
)

# fig.write_json("<path-to-directory>")
# fig.write_image("<path-to-directory>")
# fig.write_html("<path-to-directory>")
# fig.write_json("<path-to-directory>")
# fig.write_image("<path-to-directory>")
# fig.write_html("<path-to-directory>")

fig.show()

In [ ]:
import plotly.graph_objects as go

data_copy = dataset.copy()


# desired_families = ['autoins', 'smsreg', 'hiddad', 'smssend', 'umpay', 'triada', 'ewind', 'kreditspy', 'dianjin', 'traca', 'smsspy', 'dataeye', 'cryxos']
# # Filter to only 'autoins' family
# data_copy = data_copy[data_copy['family'].isin(desired_families)]


# dropping all singleton
data_copy = data_copy[data_copy['family'] == 'SINGLETON']



# Replace NaN values with 0 and non-NaN values with 1 for vpn, tor, and tor_bridge
data_copy['vpn'] = data_copy['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor'] = data_copy['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor_bridge'] = data_copy['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['proxy'] = data_copy['proxy'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['i2p'] = data_copy['i2p'].apply(lambda x: 1 if not pd.isna(x) else 0)

# set data_copy to have only columns, package, year, tor, tor_bridge, vpn, proxy
# data_copy = data_copy[['package', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
data_copy = data_copy[['family', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
# data_copy = data_copy[['hash', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]

# print(data_copy.iloc[0])

data_copy = data_copy[data_copy['year'] >= 2020]

def get_category(row):
    categories = []
    if row['tor'] == 1:
        categories.append('Tor')
    if row['tor_bridge'] == 1:
        categories.append('TorPT')
    if row['vpn'] == 1:
        categories.append('VPN')
    if row['proxy'] == 1:
        categories.append('Proxy')
    if row['i2p'] == 1:
        categories.append('I2P')
    
    if not categories:
        return "none"
    
    return ', '.join(categories)

data_copy['category'] = data_copy.apply(get_category, axis=1)

# Find unique years and categories
years = sorted(data_copy['year'].unique())
categories = data_copy['category'].unique()

year_category_map = {}

color_palette = [
    "rgba(255, 182, 193, 0.7)",  # Light Pink
    "rgba(173, 216, 230, 0.7)",  # Light Blue
    "rgba(144, 238, 144, 0.7)",  # Light Green
    "rgba(255, 218, 185, 0.7)",  # Peach Puff
    "rgba(216, 191, 216, 0.7)",  # Thistle
    "rgba(255, 228, 181, 0.7)",  # Moccasin
    "rgba(222, 184, 135, 0.7)",  # Burlywood
    "rgba(176, 224, 230, 0.7)",  # Powder Blue
    "rgba(255, 160, 122, 0.7)",  # Light Salmon
    "rgba(255, 250, 205, 0.7)",  # Lemon Chiffon
    "rgba(240, 128, 128, 0.7)",  # Light Coral
    "rgba(135, 206, 250, 0.7)",  # Light Sky Blue
    "rgba(152, 251, 152, 0.7)",  # Pale Green
    "rgba(221, 160, 221, 0.7)",  # Plum
    "rgba(46, 139, 87, 0.7)",    # Sea Green
    "rgba(255, 105, 180, 0.7)",  # Hot Pink
    "rgba(100, 149, 237, 0.7)",  # Cornflower Blue
    "rgba(60, 179, 113, 0.7)",   # Medium Sea Green
    "rgba(255, 222, 173, 0.7)",  # Navajo White
    "rgba(238, 130, 238, 0.7)",  # Violet
    "rgba(255, 69, 0, 0.7)",     # Red Orange
    "rgba(255, 192, 203, 0.7)",  # Pink
    "rgba(34, 139, 34, 0.7)",    # Forest Green
    "rgba(255, 239, 213, 0.7)",  # Papaya Whip
    "rgba(186, 85, 211, 0.7)",   # Medium Orchid
    "rgba(255, 140, 0, 0.7)",    # Dark Orange
    "rgba(70, 130, 180, 0.7)",   # Steel Blue
    "rgba(0, 191, 255, 0.7)",    # Deep Sky Blue
    "rgba(255, 248, 220, 0.7)",  # Cornsilk
    "rgba(218, 112, 214, 0.7)",  # Orchid
    "rgba(255, 165, 0, 0.7)"     # Orange
]
category_color = {category: color for category, color in zip(categories, color_palette)}

# Populate the dictionary with unique year-category combinations
for year in years:
    for category in categories:
        key = f"{year} - {category}"
        year_category_map[key] = len(year_category_map)

labels = list(year_category_map.keys())

node_colors = []
for label in labels:
    _, category = label.split(" - ")
    print(f"{category} - {category_color[category]}")
    node_colors.append(category_color[category])

# Prepare Sankey data
sources = []
targets = []
values = []
link_colors = []


# packages = data_copy["package"].unique()
families = data_copy["family"].unique()
# hashes = data_copy["hash"].unique()

start_year = data_copy['year'].min()
end_year = data_copy['year'].max()


from collections import Counter

# for family in families:
#     family_data = data_copy[data_copy['family'] == family]
#     family_years = sorted(family_data['year'].unique())

#     for i in range(len(family_years) - 1):
#         y1 = family_years[i]
#         y2 = family_years[i + 1]

#         data_y1 = family_data[family_data['year'] == y1]
#         data_y2 = family_data[family_data['year'] == y2]

#         # Count transitions between all categories in y1 and y2
#         for _, row1 in data_y1.iterrows():
#             cat1 = row1['category']

#             # Find the closest match in y2 (same hash/package logic not known, so assuming all-to-all mapping)
#             for _, row2 in data_y2.iterrows():
#                 cat2 = row2['category']

#                 # Handle in-between years
#                 for y_gap in range(y1, y2 - 1):
#                     intermediate_label = f"{y_gap} - {cat1}"
#                     next_label = f"{y_gap + 1} - {cat1}"
#                     sources.append(year_category_map[intermediate_label])
#                     targets.append(year_category_map[next_label])
#                     values.append(1)
#                     link_colors.append(category_color[cat1])

#                 # Final edge to next year
#                 source_label = f"{y1} - {cat1}"
#                 target_label = f"{y2} - {cat2}"
#                 sources.append(year_category_map[source_label])
#                 targets.append(year_category_map[target_label])
#                 values.append(1)
#                 link_colors.append(category_color[cat1])

#                 if cat1 != cat2:
#                     print(f"Transition: {family} from {y1} to {y2}: {cat1} → {cat2}")


seen_transitions_by_family = {}

for family in families:
    family_data = data_copy[data_copy['family'] == family]
    family_years = sorted(family_data['year'].unique())
    seen_transitions = set()

    for i in range(len(family_years) - 1):
        y1 = family_years[i]
        y2 = family_years[i + 1]

        data_y1 = family_data[family_data['year'] == y1]
        data_y2 = family_data[family_data['year'] == y2]

        for _, row1 in data_y1.iterrows():
            cat1 = row1['category']

            for _, row2 in data_y2.iterrows():
                cat2 = row2['category']

                # Handle in-between years
                for y_gap in range(y1, y2 - 1):
                    intermediate = (y_gap, cat1, y_gap + 1, cat1)
                    if intermediate not in seen_transitions:
                        src_label = f"{y_gap} - {cat1}"
                        tgt_label = f"{y_gap + 1} - {cat1}"

                        sources.append(year_category_map[src_label])
                        targets.append(year_category_map[tgt_label])
                        values.append(1)
                        link_colors.append(category_color[cat1])
                        seen_transitions.add(intermediate)

                final_transition = (y1, cat1, y2, cat2)
                if final_transition not in seen_transitions:
                    src_label = f"{y1} - {cat1}"
                    tgt_label = f"{y2} - {cat2}"

                    sources.append(year_category_map[src_label])
                    targets.append(year_category_map[tgt_label])
                    values.append(1)
                    link_colors.append(category_color[cat1])
                    seen_transitions.add(final_transition)

                    if cat1 != cat2:
                        print(f"Transition in {family}: {y1} {cat1} → {y2} {cat2}")

    seen_transitions_by_family[family] = seen_transitions

categories = set([label.split(" - ")[1] for label in labels])
categories = list(categories)

# Define the custom order
custom_order = ["Tor", "TorPT", "VPN", "Proxy", "I2P"]

# Helper function to get the custom order index
def get_custom_order_index(element):
    # Split by comma and find the index of each element in the custom order
    return [custom_order.index(e.strip()) for e in element.split(', ')]

# Sort by count of elements first (number of commas + 1), then by the custom order
sorted_categories = sorted(categories, key=lambda x: (len(x.split(', ')), get_custom_order_index(x)))

years = set([int(label.split(" - ")[0]) for label in labels])

bar_size_years = {}

for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    if year not in bar_size_years:
        bar_size_years[year] = {}

    if category not in bar_size_years[year]:
        idx = labels.index(label)
        bar_size_years[year][category] = max(sources.count(idx), targets.count(idx))

for year in bar_size_years:
    total = sum(bar_size_years[year].values())
    if total != 0:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = bar_size_years[year][category] / total
    else:
        for category in categories:
            if category in bar_size_years:
                bar_size_years[year].pop(category)

    diff = 0
    for category in sorted_categories:
        if category in bar_size_years[year]:
            temp = bar_size_years[year][category]
            bar_size_years[year][category] = diff
            diff += temp

    max_cat = max(bar_size_years[year].values())
    min_cat = min(bar_size_years[year].values())

    start = 0.02
    end = 0.98

    if max_cat != min_cat:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (bar_size_years[year][category] - min_cat) * (end - start) / (max_cat - min_cat)
    elif len(categories) >= 1:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (end - start) / len(categories) * sorted_categories.index(category)


y_positions = []
for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    y_positions.append(bar_size_years[year][category])


x_positions = [0.01 + (int(label.split(" - ")[0]) - min(years)) * (0.98) / (max(years) - min(years)) for label in labels]

to_remove = set(list(range(len(labels))))
to_remove -= set(sources)
to_remove -= set(targets)

change_map = {}
diff = 0
for idx in range(len(labels)):
    if idx in to_remove:
        # change_map[idx] = idx - diff
        labels.pop(idx - diff)
        x_positions.pop(idx - diff)
        y_positions.pop(idx - diff)
        node_colors.pop(idx - diff)
        diff += 1
    else:
        change_map[idx] = idx - diff

for idx in range(len(sources)):
    sources[idx] = change_map[sources[idx]]
    targets[idx] = change_map[targets[idx]]

fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        x = x_positions,
        y = y_positions,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Sankey Diagram of VPN, TOR, TorPT, Proxy, I2P Usage by Year",
    font_size=18,
    height=1200,
    width=6000
    
)

# fig.write_json("<path-to-directory>")
# fig.write_image("<path-to-directory>")
# fig.write_html("<path-to-directory>")
fig.write_json("<path-to-directory>")
fig.write_image("<path-to-directory>")
fig.write_html("<path-to-directory>")

fig.show()

In [ ]:
import plotly.graph_objects as go

data_copy = dataset.copy()


# desired_families = ['autoins', 'smsreg', 'hiddad', 'smssend', 'umpay', 'triada', 'ewind', 'kreditspy', 'dianjin', 'traca', 'smsspy', 'dataeye', 'cryxos']
# # Filter to only 'autoins' family
# data_copy = data_copy[data_copy['family'].isin(desired_families)]


# dropping all singleton
data_copy = data_copy[data_copy['family'] == 'SINGLETON']



# Replace NaN values with 0 and non-NaN values with 1 for vpn, tor, and tor_bridge
data_copy['vpn'] = data_copy['vpn'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor'] = data_copy['tor'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['tor_bridge'] = data_copy['tor_bridge'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['proxy'] = data_copy['proxy'].apply(lambda x: 1 if not pd.isna(x) else 0)
data_copy['i2p'] = data_copy['i2p'].apply(lambda x: 1 if not pd.isna(x) else 0)

# set data_copy to have only columns, package, year, tor, tor_bridge, vpn, proxy
# data_copy = data_copy[['package', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
data_copy = data_copy[['family', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]
# data_copy = data_copy[['hash', 'year', 'tor', 'tor_bridge', 'vpn', 'proxy', 'i2p']]

print(data_copy.iloc[0])

# data_copy = data_copy[data_copy['year'] >= 2019]

def get_category(row):
    categories = []
    if row['tor'] == 1:
        categories.append('Tor')
    if row['tor_bridge'] == 1:
        categories.append('TorPT')
    if row['vpn'] == 1:
        categories.append('VPN')
    if row['proxy'] == 1:
        categories.append('Proxy')
    if row['i2p'] == 1:
        categories.append('I2P')
    
    if not categories:
        return "none"
    
    return ', '.join(categories)

data_copy['category'] = data_copy.apply(get_category, axis=1)

# Find unique years and categories
years = sorted(data_copy['year'].unique())
categories = data_copy['category'].unique()

year_category_map = {}

color_palette = [
    "rgba(255, 182, 193, 0.7)",  # Light Pink
    "rgba(173, 216, 230, 0.7)",  # Light Blue
    "rgba(144, 238, 144, 0.7)",  # Light Green
    "rgba(255, 218, 185, 0.7)",  # Peach Puff
    "rgba(216, 191, 216, 0.7)",  # Thistle
    "rgba(255, 192, 203, 0.7)",  # Pink
    "rgba(222, 184, 135, 0.7)",  # Burlywood
    "rgba(176, 224, 230, 0.7)",  # Powder Blue
    "rgba(255, 160, 122, 0.7)",  # Light Salmon
    "rgba(255, 250, 205, 0.7)",  # Lemon Chiffon
    "rgba(240, 128, 128, 0.7)",  # Light Coral
    "rgba(135, 206, 250, 0.7)",  # Light Sky Blue
    "rgba(152, 251, 152, 0.7)",  # Pale Green
    "rgba(255, 228, 181, 0.7)",  # Moccasin
    "rgba(221, 160, 221, 0.7)",  # Plum
    "rgba(255, 105, 180, 0.7)",  # Hot Pink
    "rgba(100, 149, 237, 0.7)",  # Cornflower Blue
    "rgba(60, 179, 113, 0.7)",   # Medium Sea Green
    "rgba(255, 222, 173, 0.7)",  # Navajo White
    "rgba(238, 130, 238, 0.7)",  # Violet
    "rgba(255, 69, 0, 0.7)",     # Red Orange
    "rgba(0, 191, 255, 0.7)",    # Deep Sky Blue
    "rgba(34, 139, 34, 0.7)",    # Forest Green
    "rgba(255, 239, 213, 0.7)",  # Papaya Whip
    "rgba(186, 85, 211, 0.7)",   # Medium Orchid
    "rgba(255, 140, 0, 0.7)",    # Dark Orange
    "rgba(70, 130, 180, 0.7)",   # Steel Blue
    "rgba(46, 139, 87, 0.7)",    # Sea Green
    "rgba(255, 248, 220, 0.7)",  # Cornsilk
    "rgba(218, 112, 214, 0.7)",  # Orchid
    "rgba(255, 165, 0, 0.7)"     # Orange
]
category_color = {category: color for category, color in zip(categories, color_palette)}

# Populate the dictionary with unique year-category combinations
for year in years:
    for category in categories:
        key = f"{year} - {category}"
        year_category_map[key] = len(year_category_map)

labels = list(year_category_map.keys())

node_colors = []
for label in labels:
    _, category = label.split(" - ")
    node_colors.append(category_color[category])

# Prepare Sankey data
sources = []
targets = []
values = []
link_colors = []


# packages = data_copy["package"].unique()
families = data_copy["family"].unique()
# hashes = data_copy["hash"].unique()

start_year = data_copy['year'].min()
end_year = data_copy['year'].max()

# for year in years[:-1]:
#     for category in categories:
#         key = f"{year} - {category}"
#         next_key = f"{year + 1} - {category}"

#         sources.append(year_category_map[key])
#         targets.append(year_category_map[next_key])
#         values.append(1)
#         link_colors.append("rgba(255, 255, 255, 1)")

# Mann modif start


# for package in packages:
#     package_data = data_copy[data_copy['package'] == package]

#     cur_year = package_data['year'].min()

#     package_years = list(sorted(package_data['year'].unique()))

#     for year in package_years[1:]:
#         next_year = year
#         cur_category = package_data[package_data['year'] == cur_year].iloc[0]['category']
#         next_category = package_data[package_data['year'] == next_year].iloc[0]['category']

#         while (cur_year < year-1):
#             source_label = f"{cur_year} - {cur_category}"
#             target_label = f"{cur_year + 1} - {cur_category}"

#             sources.append(year_category_map[source_label])
#             targets.append(year_category_map[target_label])
#             values.append(1)
#             link_colors.append(category_color[cur_category])

#             cur_year += 1

#         source_label = f"{cur_year} - {cur_category}"
#         target_label = f"{next_year} - {next_category}"

#         sources.append(year_category_map[source_label])
#         targets.append(year_category_map[target_label])
#         link_colors.append(category_color[cur_category])
#         values.append(1)

#         if cur_category != next_category:
#             print(f"MIL GAYA: {package}, {cur_year}, {next_year}, |{cur_category}| |{next_category}|")

#         cur_year = year


# # This is for plotting all families FAMILY PLOT START
for family in families:
    family_data = data_copy[data_copy['family'] == family]

    cur_year = family_data['year'].min()
    family_years = list(sorted(family_data['year'].unique()))

    for year in family_years[1:]:
        next_year = year
        cur_category = family_data[family_data['year'] == cur_year].iloc[0]['category']
        next_category = family_data[family_data['year'] == next_year].iloc[0]['category']

        while (cur_year < year-1):
            source_label = f"{cur_year} - {cur_category}"
            target_label = f"{cur_year + 1} - {cur_category}"

            sources.append(year_category_map[source_label])
            targets.append(year_category_map[target_label])
            values.append(1)
            link_colors.append(category_color[cur_category])

            cur_year += 1

        source_label = f"{cur_year} - {cur_category}"
        target_label = f"{next_year} - {next_category}"

        sources.append(year_category_map[source_label])
        targets.append(year_category_map[target_label])
        link_colors.append(category_color[cur_category])
        values.append(1)

        if cur_category != next_category:
            print(f"MIL GAYA: {family}, {cur_year}, {next_year}, |{cur_category}| |{next_category}|")

        cur_year = year
# # FAMILY PLOT END

# AUTOINS PLOT START

# for h in hashes:
#     hash_data = data_copy[data_copy['hash'] == h]

#     cur_year = hash_data['year'].min()
#     hash_years = list(sorted(hash_data['year'].unique()))
#     print(f"{h}: {hash_years}")  # Debug: which hashes span multiple years?

#     for year in hash_years[1:]:
#         next_year = year
#         cur_category = hash_data[hash_data['year'] == cur_year].iloc[0]['category']
#         next_category = hash_data[hash_data['year'] == next_year].iloc[0]['category']

#         while (cur_year < year-1):
#             source_label = f"{cur_year} - {cur_category}"
#             target_label = f"{cur_year + 1} - {cur_category}"

#             sources.append(year_category_map[source_label])
#             targets.append(year_category_map[target_label])
#             values.append(1)
#             link_colors.append(category_color[cur_category])

#             cur_year += 1

#         source_label = f"{cur_year} - {cur_category}"
#         target_label = f"{next_year} - {next_category}"

#         sources.append(year_category_map[source_label])
#         targets.append(year_category_map[target_label])
#         link_colors.append(category_color[cur_category])
#         values.append(1)

#         if cur_category != next_category:
#             print(f"MIL GAYA: {h}, {cur_year}, {next_year}, |{cur_category}| |{next_category}|")

#         cur_year = year


# AUTOINS PLOT END


# Mann modif end


    # if year != years[-1]:
    #     row = package_data[package_data['year'] == cur_year].iloc[0]
    #     while (cur_year != years[-1]):
    #         next_year = cur_year + 1
    #         cur_category = row['category']
    #         next_category = row['category']

    #         source_label = f"{cur_year} - {cur_category}"
    #         target_label = f"{next_year} - {next_category}"

    #         sources.append(year_category_map[source_label])
    #         targets.append(year_category_map[target_label])
    #         values.append(1)
    #         link_colors.append(category_color[cur_category])

    #         cur_year += 1

# Create Sankey diagram

categories = set([label.split(" - ")[1] for label in labels])
categories = list(categories)

# Define the custom order
custom_order = ["Tor", "TorPT", "VPN", "Proxy", "I2P"]

# Helper function to get the custom order index
def get_custom_order_index(element):
    # Split by comma and find the index of each element in the custom order
    return [custom_order.index(e.strip()) for e in element.split(', ')]

# Sort by count of elements first (number of commas + 1), then by the custom order
sorted_categories = sorted(categories, key=lambda x: (len(x.split(', ')), get_custom_order_index(x)))

years = set([int(label.split(" - ")[0]) for label in labels])

bar_size_years = {}

for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    if year not in bar_size_years:
        bar_size_years[year] = {}

    if category not in bar_size_years[year]:
        idx = labels.index(label)
        bar_size_years[year][category] = max(sources.count(idx), targets.count(idx))

for year in bar_size_years:
    total = sum(bar_size_years[year].values())
    if total != 0:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = bar_size_years[year][category] / total
    else:
        for category in categories:
            if category in bar_size_years:
                bar_size_years[year].pop(category)

    diff = 0
    for category in sorted_categories:
        if category in bar_size_years[year]:
            temp = bar_size_years[year][category]
            bar_size_years[year][category] = diff
            diff += temp

    max_cat = max(bar_size_years[year].values())
    min_cat = min(bar_size_years[year].values())

    start = 0.02
    end = 0.98

    if max_cat != min_cat:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (bar_size_years[year][category] - min_cat) * (end - start) / (max_cat - min_cat)
    elif len(categories) >= 1:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (end - start) / len(categories) * sorted_categories.index(category)


y_positions = []
for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    y_positions.append(bar_size_years[year][category])


x_positions = [0.01 + (int(label.split(" - ")[0]) - min(years)) * (0.98) / (max(years) - min(years)) for label in labels]

to_remove = set(list(range(len(labels))))
to_remove -= set(sources)
to_remove -= set(targets)

change_map = {}
diff = 0
for idx in range(len(labels)):
    if idx in to_remove:
        # change_map[idx] = idx - diff
        labels.pop(idx - diff)
        x_positions.pop(idx - diff)
        y_positions.pop(idx - diff)
        node_colors.pop(idx - diff)
        diff += 1
    else:
        change_map[idx] = idx - diff

for idx in range(len(sources)):
    sources[idx] = change_map[sources[idx]]
    targets[idx] = change_map[targets[idx]]

fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        x = x_positions,
        y = y_positions,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Sankey Diagram of VPN, TOR, TorPT, Proxy Usage by Year",
    font_size=18,
    height=1200,
    width=6000
    
)

fig.write_json("<path-to-directory>")
fig.write_image("<path-to-directory>")
fig.write_html("<path-to-directory>")
# fig.write_json("<path-to-directory>")
# fig.write_image("<path-to-directory>")
# fig.write_html("<path-to-directory>")

fig.show()

In [ ]:
labels = fig.data[0].node.label

categories = set([label.split(" - ")[1] for label in labels])
categories = list(categories)

# Define the custom order
custom_order = ["Tor", "TorPT", "VPN", "Proxy", "I2P"]

# Helper function to get the custom order index
def get_custom_order_index(element):
    # Split by comma and find the index of each element in the custom order
    return [custom_order.index(e.strip()) for e in element.split(', ')]

# Sort by count of elements first (number of commas + 1), then by the custom order
sorted_categories = sorted(categories, key=lambda x: (len(x.split(', ')), get_custom_order_index(x)))

years = set([int(label.split(" - ")[0]) for label in labels])

bar_size_years = {}

source_nodes = fig.data[0].link.source
target_nodes = fig.data[0].link.target

for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    if year not in bar_size_years:
        bar_size_years[year] = {}

    if category not in bar_size_years[year]:
        idx = labels.index(label)
        bar_size_years[year][category] = max(source_nodes.count(idx), target_nodes.count(idx))

for year in bar_size_years:
    total = sum(bar_size_years[year].values())
    for category in bar_size_years[year]:
        bar_size_years[year][category] = bar_size_years[year][category] / total

    diff = 0
    for category in sorted_categories:
        if category in bar_size_years[year]:
            temp = bar_size_years[year][category]
            bar_size_years[year][category] = diff
            diff += temp

    max_cat = max(bar_size_years[year].values())
    min_cat = min(bar_size_years[year].values())

    start = 0.02
    end = 0.98

    if len(bar_size_years[year].keys()) > 1:
        for category in bar_size_years[year]:
            bar_size_years[year][category] = start + (bar_size_years[year][category] - min_cat) * (end - start) / (max_cat - min_cat)


y_positions = []
for label in labels:
    year = int(label.split(" - ")[0])
    category = label.split(" - ")[1]
    y_positions.append(bar_size_years[year][category])


x_positions = [0.01 + (int(label.split(" - ")[0]) - min(years)) * (0.98) / (max(years) - min(years)) for label in labels]
# y_positions = [0.01 + (sorted_categories.index(label.split(" - ")[1])) * (0.84) / (len(categories) - 1) for label in labels]

In [ ]:
fig.data[0].node.x = x_positions
fig.data[0].node.y = y_positions

fig.update_layout(
    width=6000,
    height=1200
)

fig.write_json("<path-to-directory>")
fig.write_image("<path-to-directory>")
fig.write_html("<path-to-directory>")

fig.show()

In [ ]:
source_copy = []
target_copy = []
value_copy = []
link_color_copy = []

In [ ]:
proxy_idx = set()
for idx in range(len(labels)):
    if labels[idx].split(" - ")[1] == "Proxy":
        proxy_idx.add(idx)

print(proxy_idx)

In [ ]:
remove_thres = 0

removed = {k: 0 for k in proxy_idx}
for idx in range(len(sources)):
    if sources[idx] in proxy_idx and targets[idx] in proxy_idx:
        removed[sources[idx]] += 1
        if removed[sources[idx]] == remove_thres:
            removed[sources[idx]] = 0
            source_copy.append(sources[idx])
            target_copy.append(targets[idx])
            value_copy.append(values[idx])
            link_color_copy.append(link_colors[idx])
        continue

    source_copy.append(sources[idx])
    target_copy.append(targets[idx])
    value_copy.append(values[idx])
    link_color_copy.append(link_colors[idx])

In [ ]:
source_copy.count(0)

In [ ]:
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        x = x_positions,
        y = y_positions,
        color=node_colors,
        align="right"
    ),
    link=dict(
        source=source_copy,
        target=target_copy,
        value=value_copy,
        color=link_color_copy
    )
)])

fig.update_layout(font_size=26, height=1000, width=4000, font_weight=1000)

# after running the above cell please run the last sankey generation again

In [ ]:
fig.write_image("<path-to-directory>")
fig.write_html("<path-to-directory>")
fig.write_json("<path-to-directory>")

In [ ]:
y_positions

In [ ]:
fig.write_image("<path-to-directory>")

In [ ]:
labels_copy = labels[:]
x_copy = x_positions[:]
y_copy = y_positions[:]
color_copy = node_colors[:]
sources_copy = sources[:]
targets_copy = targets[:]

In [ ]:
to_remove = set(list(range(len(labels_copy))))
to_remove -= set(sources)
to_remove -= set(targets)

In [ ]:
change_map = {}
diff = 0
for idx in range(len(labels_copy)):
    if idx in to_remove:
        # change_map[idx] = idx - diff
        labels_copy.pop(idx - diff)
        x_copy.pop(idx - diff)
        y_copy.pop(idx - diff)
        color_copy.pop(idx - diff)
        diff += 1
    else:
        change_map[idx] = idx - diff


In [ ]:
for idx in range(len(sources_copy)):
    sources_copy[idx] = change_map[sources_copy[idx]]
    targets_copy[idx] = change_map[targets_copy[idx]]

In [ ]:
node_trace = go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels_copy,
        x = x_copy,
        y = y_copy,
        color=color_copy,
        align="right"
    ),
    link=dict(
        source=sources_copy,
        target=targets_copy,
        value=values,
        color=link_colors
    )
)


In [ ]:
len(labels_copy)

In [ ]:
max(x_positions)

In [ ]:
fig = pio.read_json("<path-to-directory>")

In [ ]:
fig = go.Figure(node_trace, layout=go.Layout(title_text="Sankey Diagram of VPN, TOR, TorPT, Proxy Usage by Year", font_size=14, height=1200, width=2000))

In [ ]:
fig.show()

In [ ]:
import plotly.io as pio

fig = pio.read_json("<path-to-directory>")

In [ ]:
fig = fig.update_layout(height=1200, width=2000)

In [ ]:
fig.write_image("<path-to-directory>")

In [ ]:
original_label = fig.data[0].node.label

In [ ]:
possible_x = years

In [ ]:
year_category_map

In [ ]:
x_positions = [(int(label.split(" - ")[0]) - min(possible_x) + 1) / (len(possible_x) + 1) for label in original_label]

categories = set([label.split(" - ")[1] for label in original_label])

In [ ]:
categories = list(categories)

# Define the custom order
custom_order = ["Tor", "TorPT", "VPN", "Proxy"]

# Helper function to get the custom order index
def get_custom_order_index(element):
    # Split by comma and find the index of each element in the custom order
    return [custom_order.index(e.strip()) for e in element.split(', ')]

# Sort by count of elements first (number of commas + 1), then by the custom order
sorted_categories = sorted(categories, key=lambda x: (len(x.split(', ')), get_custom_order_index(x)))

# Print the sorted list
print(sorted_categories)

In [ ]:
possible_y = sorted_categories

In [ ]:
y_positions = [(sorted_categories.index(label.split(" - ")[1]) + 1) / (len(sorted_categories) + 1) for label in original_label]

In [ ]:
fig.data[0].arrangement="snap"

In [ ]:
positions = {}
for label, x, y in zip(original_label, x_positions, y_positions):
    positions[label] = (x, y)

print(positions)

In [ ]:
fig.data[0].node.align = "right"

In [ ]:
fig.data[0].node.x = [0]*len(original_label)
fig.data[0].node.y = [0]*len(original_label)

# a = [0] * len(original_label)
# a[3] = 0.1

# fig.data[0].node.x = a

In [ ]:
fig.data[0].node.y = y_positions

In [ ]:
fig.write_json("<path-to-directory>")

In [ ]:
fig.data[0].node.x = x_positions

In [ ]:
len(labels)

In [ ]:
list(zip(x_positions, y_positions))

In [ ]:
fig.show()

In [ ]:
labels = ["2019 - proxy","2019 - vpn, proxy","2019 - vpn","2019 - tor_bridge, vpn","2019 - tor_bridge, vpn, proxy","2019 - tor","2019 - tor, proxy","2019 - tor_bridge","2019 - tor, tor_bridge, proxy","2019 - tor_bridge, proxy","2019 - tor, vpn, proxy","2019 - tor, tor_bridge, vpn","2019 - tor, vpn","2020 - proxy","2020 - vpn, proxy","2020 - vpn","2020 - tor_bridge, vpn","2020 - tor_bridge, vpn, proxy","2020 - tor","2020 - tor, proxy","2020 - tor_bridge","2020 - tor, tor_bridge, proxy","2020 - tor_bridge, proxy","2020 - tor, vpn, proxy","2020 - tor, tor_bridge, vpn","2020 - tor, vpn","2021 - proxy","2021 - vpn, proxy","2021 - vpn","2021 - tor_bridge, vpn","2021 - tor_bridge, vpn, proxy","2021 - tor","2021 - tor, proxy","2021 - tor_bridge","2021 - tor, tor_bridge, proxy","2021 - tor_bridge, proxy","2021 - tor, vpn, proxy","2021 - tor, tor_bridge, vpn","2021 - tor, vpn","2022 - proxy","2022 - vpn, proxy","2022 - vpn","2022 - tor_bridge, vpn","2022 - tor_bridge, vpn, proxy","2022 - tor","2022 - tor, proxy","2022 - tor_bridge","2022 - tor, tor_bridge, proxy","2022 - tor_bridge, proxy","2022 - tor, vpn, proxy","2022 - tor, tor_bridge, vpn","2022 - tor, vpn","2023 - proxy","2023 - vpn, proxy","2023 - vpn","2023 - tor_bridge, vpn","2023 - tor_bridge, vpn, proxy","2023 - tor","2023 - tor, proxy","2023 - tor_bridge","2023 - tor, tor_bridge, proxy","2023 - tor_bridge, proxy","2023 - tor, vpn, proxy","2023 - tor, tor_bridge, vpn","2023 - tor, vpn","2024 - proxy","2024 - vpn, proxy","2024 - vpn","2024 - tor_bridge, vpn","2024 - tor_bridge, vpn, proxy","2024 - tor","2024 - tor, proxy","2024 - tor_bridge","2024 - tor, tor_bridge, proxy","2024 - tor_bridge, proxy","2024 - tor, vpn, proxy","2024 - tor, tor_bridge, vpn","2024 - tor, vpn"]

for i in range(len(labels)):
    year = labels[i].split(" - ")[0]
    category = labels[i].split(" - ")[1]
    categories = category.split(", ")
    for j in range(len(categories)):
        if categories[j] == "vpn":
            categories[j] = "VPN"
        elif categories[j] == "tor":
            categories[j] = "Tor"
        elif categories[j] == "tor_bridge":
            categories[j] = "TorPT"
        elif categories[j] == "proxy":
            categories[j] = "Proxy"

    labels[i] = f"{year} - {', '.join(categories)}"

In [ ]:
fig.update_layout(font_size=18)

In [ ]:
fig.write_image("<path-to-directory>")

In [ ]:
fig.write_html("<path-to-directory>")

In [ ]:
fig.write_json("<path-to-directory>")

In [ ]:
data_copy

In [ ]:
wordlist_file = "<path-to-directory>"

wordlist = set()

with open(wordlist_file) as f:
    for line in f:
        wordlist.add(line.strip())



In [ ]:
def is_obfuscated(service):
    service = service.lower().split(".")
    for service_word in service:
        if service_word in wordlist:
            continue
        if service_word in ["io", "ui", "os"]:
            continue
        if len(service_word) >= 5:
            continue
        return True
    
    return False

In [ ]:
dataset["obfuscated"] = dataset["package"].apply(is_obfuscated)

In [ ]:
# print table of package, count and obfuscated
package_count = dataset['package'].value_counts()
# add a column to package_count representing obfuscation
package_count["Obfuscated"] = dataset[dataset["obfuscated"] == True].shape[0]
package_count

In [ ]:
# value count of obfuscated packages
dataset[dataset["obfuscated"] == True]["package"].value_counts()

In [ ]:
dataset["obfuscated"].value_counts()

In [ ]:
unobfuscated = set([i for i in dataset["package"] if not is_obfuscated(i)])

In [ ]:
len(unobfuscated)

In [ ]:


st = set()
for i in dataset[dataset["obfuscated"] == True]["package"]:
    st.update(set(i.split(".")))

st = set([i.lower() for i in st if 3 <= len(i) <= 5])

In [ ]:
len(st)

In [ ]:
set(dataset["user"])

In [ ]:
import os

dataset_years = dataset[dataset["year"].isin([2015,2016,2018,2019])]

vt_folder = "<path-to-directory>"

# Iterate through the dataset
for index, row in dataset_years.iterrows():
    year = row['year']
    month = row['month']
    hash_value = row['hash']
    
    # Create a folder for the year if it doesn't exist
    year_folder = os.path.join('vt_test', str(year))
    if not os.path.exists(year_folder):
        os.makedirs(year_folder)

    if os.path.exists(os.path.join(vt_folder, str(year), str(month), f"{hash_value.lower()}.json")):
        hash_value = hash_value.lower()
    elif os.path.exists(os.path.join(vt_folder, str(year), str(month), f"{hash_value.upper()}.json")):
        hash_value = hash_value.upper()
    else:
        print(f"did not find {hash_value}")
    
    # Create a file named {month}.txt and write the hash in it
    month_file = os.path.join(year_folder, f'{month}.txt')
    with open(month_file, 'a') as file:
        file.write(hash_value + '\n')

for year in os.listdir("vt_test"):
    for month in os.listdir(os.path.join("vt_test", year)):
        with open(os.path.join("vt_test", year, month), "a+") as f:
            done = set()
            for i in f:
                done.add(i.strip())

            counter = 0
            for i in os.listdir(os.path.join(vt_folder, year, month.split(".")[0])):
                if counter == 700:
                    break
                if i not in done:
                    f.write(i + "\n")
                    counter += 1

In [ ]:
tor_validated = set(dataset[dataset["tor"].notna()]["hash"].str.lower())
pt_validated = set(dataset[dataset["tor_bridge"].notna()]["hash"].str.lower())
vpn_validated = set(dataset[dataset["vpn"].notna()]["hash"].str.lower())
proxy_validated = set(dataset[dataset["proxy"].notna()]["hash"].str.lower())

In [ ]:
filtered_folder = "<path-to-directory>"

tor_vt = set()
pt_vt = set()
vpn_vt = set()
proxy_vt = set()

for year in os.listdir(filtered_folder):
    for month in os.listdir(os.path.join(filtered_folder, year)):
        with open(os.path.join(filtered_folder, year, month, "TOR.txt")) as f:
            for i in f:
                tor_vt.add(i.strip().split(",")[0].split(".")[0].strip().lower())

        with open(os.path.join(filtered_folder, year, month, "TOR_BRIDGE.txt")) as f:
            for i in f:
                pt_vt.add(i.strip().split(",")[0].split(".")[0].strip().lower())

        with open(os.path.join(filtered_folder, year, month, "PROXY.txt")) as f:
            for i in f:
                vpn_vt.add(i.strip().split(",")[0].split(".")[0].strip().lower())

        with open(os.path.join(filtered_folder, year, month, "VPN.txt")) as f:
            for i in f:
                proxy_vt.add(i.strip().split(",")[0].split(".")[0].strip().lower())

In [ ]:
def get_confusion_matrix(true, pred):
    TP = len(true.intersection(pred))
    FP = len(pred - true)
    FN = len(true - pred)
    
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)

    return TP, FP, FN, precision, recall

TP = len(tor_vt.intersection(tor_validated))
FP = len(tor_vt - tor_validated)
FN = len(tor_validated - tor_vt)
TN = len(tor_validated.union(tor_vt)) # is this correct???

In [ ]:
print("Tor VT Filtering Stats:")
TP, FP, FN, precision, recall = get_confusion_matrix(tor_validated, tor_vt)
print(f"TP: {TP}, FP: {FP}, FN: {FN}")
print(f"Precision: {precision}, Recall: {recall}")
print()
print("TorPT VT Filtering Stats:")
TP, FP, FN, precision, recall = get_confusion_matrix(pt_validated, pt_vt)
print(f"TP: {TP}, FP: {FP}, FN: {FN}")
print(f"Precision: {precision}, Recall: {recall}")
print()
print("VPN VT Filtering Stats:")
TP, FP, FN, precision, recall = get_confusion_matrix(vpn_validated, vpn_vt)
print(f"TP: {TP}, FP: {FP}, FN: {FN}")
print(f"Precision: {precision}, Recall: {recall}")
print()
print("Proxy VT Filtering Stats:")
TP, FP, FN, precision, recall = get_confusion_matrix(proxy_validated, proxy_vt)
print(f"TP: {TP}, FP: {FP}, FN: {FN}")
print(f"Precision: {precision}, Recall: {recall}")
print()
print("Overall VT Filtering Stats")
TP, FP, FN, precision, recall = get_confusion_matrix(tor_validated.union(pt_validated).union(vpn_validated).union(proxy_validated), tor_vt.union(pt_vt).union(vpn_vt).union(proxy_vt))
print(f"TP: {TP}, FP: {FP}, FN: {FN}")
print(f"Precision: {precision}, Recall: {recall}")


In [ ]:
len(tor_validated.union(pt_validated).union(vpn_validated).union(proxy_validated))

In [ ]:
dataset = pd.read_csv("<path-to-directory>")

In [ ]:
duration_ranges = pd.read_csv("<path-to-directory>")

In [ ]:
# Calculate the cumulative count
duration_ranges['Cumulative Count'] = duration_ranges['Count'].cumsum()

# Convert cumulative count to percentage
total_count = duration_ranges['Cumulative Count'].iloc[-1]
duration_ranges['Cumulative Percentage'] = (duration_ranges['Cumulative Count'] / total_count) * 100

# Plot the CDF with percentage
plt.figure(figsize=(6, 3.5))
plt.step(duration_ranges['Range End'], duration_ranges['Cumulative Percentage'], where='post', 
         label='Cumulative Count (in %)', color='blue', linewidth=1.75)

# Add labels and title
plt.xlabel('Time Duration of Connections (in Sec)', fontsize=14, fontweight='bold')
plt.ylabel('Cumulative Count (in %)', fontsize=14, fontweight='bold')
# plt.title('Cumulative Distribution Function (CDF)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
# plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("<path-to-directory>")

corresponding CDF to the above generated ecdf

In [ ]:
# data_dynamic = {
#     "Year": [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
#     "Apps Ran": [30, 124, 79, 126, 119, 176, 281, 630, 359, 3185, 2387, 57, 325],
#     "Apps Generated PCAP": [0, 0, 0, 2, 31, 40, 96, 403, 255, 2445, 1807, 48, 238]
# }

data_dynamic = {
    "Year": [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    "Apps Ran": [30, 314, 235, 357, 338, 529, 1176, 1558, 1124, 4040, 2769, 720, 396, 28],
    "Apps Generated PCAP": [0, 0, 0, 4, 94, 231, 461, 752, 610, 2582, 1204, 305, 192, 10]
}

percentage_apps_ran = [data_dynamic["Apps Generated PCAP"][i] / data_dynamic["Apps Ran"][i] * 100 for i in range(len(data_dynamic["Year"]))]

# Create a DataFrame
df_dynamic = pd.DataFrame(data_dynamic)

# Plot the data
plt.figure(figsize=(10, 3))
plt.plot(df_dynamic["Year"], percentage_apps_ran, marker='o', linestyle='--', label="Apps Ran", color="blue", linewidth=2)

# Add labels, title, and legend
plt.xlabel("Year", fontsize=18, fontweight='bold')
plt.ylabel("Success Rate (in %)", fontsize=14, fontweight='bold')
# plt.title("Apps Ran vs. Apps Generated PCAP (2012-2024)", fontsize=14)
# plt.legend(fontsize=16)
plt.grid(True, linestyle="--", alpha=0.7)
plt.xticks(fontsize=14, fontweight='bold')
plt.xticks(df_dynamic["Year"], fontsize=14, fontweight='bold')
plt.yticks(list(range(0, 101, 10)), fontsize=14, fontweight='bold')
# plt.yscale("log")
plt.tight_layout()

plt.savefig("<path-to-directory>")

# Show the plot
plt.show()

In [ ]:
len(dataset[dataset["vpn"].notna()])

In [ ]:
df_hehehe = pd.read_csv("<path-to-directory>")

In [ ]:
# Ensure 'count' is treated as numeric
df_hehehe['count'] = pd.to_numeric(df_hehehe['count'], errors='coerce')

# Drop rows with NaN values in 'count'
df = df_hehehe.dropna(subset=['count'])

# Sort by 'count' in descending order
df_sorted = df_hehehe.sort_values(by='count', ascending=False)

# # Take the top 10 entries
# top_10 = df_sorted.head(10)

# Take the top 10 entries and make a copy
top_10 = df_sorted.head(10).copy()

# Extract last segment of permission names
top_10['cert'] = top_10['cert'].apply(lambda x: x.split('.')[-1])

# Formatter function to convert to 'k'
def thousands_formatter(x, pos):
    return f'{int(x/1000)}'


plt.figure(figsize=(6, 4))  # wider figure
plt.plot(top_10['cert'], top_10['count'], linestyle='--', marker='o', linewidth=2)

plt.xticks(rotation=30, ha='right', fontweight='bold', fontsize=9)  # Make x-axis ticks bold
plt.yticks(fontweight='bold')  # Make y-axis ticks bold

# plt.xticks(rotation=45, ha='right')  # rotate and right-align labels
plt.xlabel('Permissions', fontsize=10, fontweight='bold')
plt.ylabel('Count of APKs (in k)', fontsize=10, fontweight='bold')
plt.gca().yaxis.set_major_formatter(FuncFormatter(thousands_formatter)) 
plt.grid(True, linestyle="--", alpha=0.7)

plt.tight_layout()

plt.savefig("<path-to-directory>")
plt.show()
plt.close()

# Ecdf plotting

In [ ]:
# Load CSV file
# df = pd.read_csv("<path-to-directory>")

df = pd.read_csv("<path-to-directory>")
# Filter valid time values
df = df[df["Time_to_CnC"].notna()]
df = df[df["Status"] == "Success"]

times = df["Time_to_CnC"].sort_values().to_numpy()
ecdf = np.arange(1, len(times)+1) / len(times)
ecdf_percent = ecdf * 100
# Plot ECDF
plt.figure(figsize=(6, 3.5))
plt.plot(times, ecdf_percent, marker='.', linestyle='none', color='blue')
plt.xlabel("Time to First CC Packet (in Sec)", fontsize=14, fontweight='bold')
plt.ylabel("Cumulative Count (in %)", fontsize=14, fontweight='bold')
# plt.title("ECDF of Time to First C&C Packet", fontsize=22, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
# Set X-axis range and ticks
plt.xlim(0, 160)
plt.xticks(np.arange(0, 161, 20), fontsize=14, fontweight='bold')  # tick every 20s
plt.yticks(fontsize=14, fontweight='bold')

# Save and show
plt.savefig("<path-to-directory>")
plt.show()

In [ ]:
# Load CSV file
# df = pd.read_csv("<path-to-directory>")

df = pd.read_csv("<path-to-directory>")
# Filter valid time values
df = df[df["Time_to_CnC"].notna()]
df = df[df["Status"] == "Success"]
df = df[df["CC"] == "VPN"]

times = df["Time_to_CnC"].sort_values().to_numpy()
ecdf = np.arange(1, len(times)+1) / len(times)
ecdf_percent = ecdf * 100
# Plot ECDF
plt.figure(figsize=(6, 3.5))
plt.plot(times, ecdf_percent, marker='.', linestyle='none', color='blue')
plt.xlabel("Time to First CC Packet (in Sec)", fontsize=14, fontweight='bold')
plt.ylabel("Cumulative Count (in %)", fontsize=14, fontweight='bold')
# plt.title("ECDF of Time to First C&C Packet", fontsize=22, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
# Set X-axis range and ticks
plt.xlim(0, 160)
plt.xticks(np.arange(0, 161, 20), fontsize=14, fontweight='bold')  # tick every 20s
plt.yticks(fontsize=14, fontweight='bold')

# Save and show
plt.savefig("<path-to-directory>")
plt.show()

In [ ]:
# Load CSV file
# df = pd.read_csv("<path-to-directory>")

df = pd.read_csv("<path-to-directory>")
# Filter valid time values
df = df[df["Time_to_CnC"].notna()]
df = df[df["Status"] == "Success"]
df = df[df["CC"] == "TOR"]

times = df["Time_to_CnC"].sort_values().to_numpy()
ecdf = np.arange(1, len(times)+1) / len(times)
ecdf_percent = ecdf * 100
# Plot ECDF
plt.figure(figsize=(6, 3.5))
plt.plot(times, ecdf_percent, marker='.', linestyle='none', color='blue')
plt.xlabel("Time to First CC Packet (in Sec)", fontsize=14, fontweight='bold')
plt.ylabel("Cumulative Count (in %)", fontsize=14, fontweight='bold')
# plt.title("ECDF of Time to First C&C Packet", fontsize=22, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
# Set X-axis range and ticks
plt.xlim(0, 160)
plt.xticks(np.arange(0, 161, 20), fontsize=14, fontweight='bold')  # tick every 20s
plt.yticks(fontsize=14, fontweight='bold')

# Save and show
plt.savefig("<path-to-directory>")
plt.show()

In [ ]:
# Load CSV file
# df = pd.read_csv("<path-to-directory>")

df = pd.read_csv("<path-to-directory>")
# Filter valid time values
df = df[df["Time_to_CnC"].notna()]
df = df[df["Status"] == "Success"]
df = df[df["CC"] == "PROXY"]

times = df["Time_to_CnC"].sort_values().to_numpy()
ecdf = np.arange(1, len(times)+1) / len(times)
ecdf_percent = ecdf * 100
# Plot ECDF
plt.figure(figsize=(6, 3.5))
plt.plot(times, ecdf_percent, marker='.', linestyle='none', color='blue')
plt.xlabel("Time to First CC Packet (in Sec)", fontsize=14, fontweight='bold')
plt.ylabel("Cumulative Count (in %)", fontsize=14, fontweight='bold')
# plt.title("ECDF of Time to First C&C Packet", fontsize=22, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
# Set X-axis range and ticks
plt.xlim(0, 160)
plt.xticks(np.arange(0, 161, 20), fontsize=14, fontweight='bold')  # tick every 20s
plt.yticks(fontsize=14, fontweight='bold')

# Save and show
plt.savefig("<path-to-directory>")
plt.show()

# CDF plot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load CSV
# df = pd.read_csv("<path-to-directory>")
df = pd.read_csv("<path-to-directory>")
df = df[df["Time_to_CnC"].notna()]
df = df[df["Status"] == "Success"]

# Bin the data (e.g., 0–1 sec, 1–2 sec, ..., or choose bin width)
bin_edges = np.arange(0, df["Time_to_CnC"].max() + 1, 1)  # 1-second bins
df['Bin'] = pd.cut(df["Time_to_CnC"], bins=bin_edges, right=False)

# Count entries per bin
duration_ranges = df['Bin'].value_counts().sort_index().reset_index()
duration_ranges.columns = ['Range', 'Count']

# Extract upper edge of each range for plotting
duration_ranges['Range End'] = duration_ranges['Range'].apply(lambda x: x.right)

# Cumulative percentage
duration_ranges['Cumulative Count'] = duration_ranges['Count'].cumsum()
total_count = duration_ranges['Cumulative Count'].iloc[-1]
duration_ranges['Cumulative Percentage'] = (duration_ranges['Cumulative Count'] / total_count) * 100

# Plot CDF
plt.figure(figsize=(6, 3.5))
plt.step(duration_ranges['Range End'], duration_ranges['Cumulative Percentage'],
         where='post', label='CDF (in %)', color='blue', linewidth=1.75)

plt.xlabel('Time Duration of Connections (in Sec)', fontsize=14, fontweight='bold')
plt.ylabel('Cumulative Count (in %)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.tight_layout()

# Save
plt.savefig("<path-to-directory>")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load CSV
# df = pd.read_csv("<path-to-directory>")
df = pd.read_csv("<path-to-directory>")
df = df[df["Time_to_CnC"].notna()]
df = df[df["Status"] == "Success"]
df = df[df["CC"] == "VPN"]

# Bin the data (e.g., 0–1 sec, 1–2 sec, ..., or choose bin width)
bin_edges = np.arange(0, df["Time_to_CnC"].max() + 1, 1)  # 1-second bins
df['Bin'] = pd.cut(df["Time_to_CnC"], bins=bin_edges, right=False)

# Count entries per bin
duration_ranges = df['Bin'].value_counts().sort_index().reset_index()
duration_ranges.columns = ['Range', 'Count']

# Extract upper edge of each range for plotting
duration_ranges['Range End'] = duration_ranges['Range'].apply(lambda x: x.right)

# Cumulative percentage
duration_ranges['Cumulative Count'] = duration_ranges['Count'].cumsum()
total_count = duration_ranges['Cumulative Count'].iloc[-1]
duration_ranges['Cumulative Percentage'] = (duration_ranges['Cumulative Count'] / total_count) * 100

# Plot CDF
plt.figure(figsize=(6, 3.5))
plt.step(duration_ranges['Range End'], duration_ranges['Cumulative Percentage'],
         where='post', label='CDF (in %)', color='blue', linewidth=1.75)

plt.xlabel('Time Duration of Connections (in Sec)', fontsize=14, fontweight='bold')
plt.ylabel('Cumulative Count (in %)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.tight_layout()

# Save
plt.savefig("<path-to-directory>")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load CSV
# df = pd.read_csv("<path-to-directory>")
df = pd.read_csv("<path-to-directory>")
df = df[df["Time_to_CnC"].notna()]
df = df[df["Status"] == "Success"]
df = df[df["CC"] == "TOR"]

# Bin the data (e.g., 0–1 sec, 1–2 sec, ..., or choose bin width)
bin_edges = np.arange(0, df["Time_to_CnC"].max() + 1, 1)  # 1-second bins
df['Bin'] = pd.cut(df["Time_to_CnC"], bins=bin_edges, right=False)

# Count entries per bin
duration_ranges = df['Bin'].value_counts().sort_index().reset_index()
duration_ranges.columns = ['Range', 'Count']

# Extract upper edge of each range for plotting
duration_ranges['Range End'] = duration_ranges['Range'].apply(lambda x: x.right)

# Cumulative percentage
duration_ranges['Cumulative Count'] = duration_ranges['Count'].cumsum()
total_count = duration_ranges['Cumulative Count'].iloc[-1]
duration_ranges['Cumulative Percentage'] = (duration_ranges['Cumulative Count'] / total_count) * 100

# Plot CDF
plt.figure(figsize=(6, 3.5))
plt.step(duration_ranges['Range End'], duration_ranges['Cumulative Percentage'],
         where='post', label='CDF (in %)', color='blue', linewidth=1.75)

plt.xlabel('Time Duration of Connections (in Sec)', fontsize=14, fontweight='bold')
plt.ylabel('Cumulative Count (in %)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.tight_layout()

# Save
plt.savefig("<path-to-directory>")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load CSV
# df = pd.read_csv("<path-to-directory>")
df = pd.read_csv("<path-to-directory>")
df = df[df["Time_to_CnC"].notna()]
df = df[df["Status"] == "Success"]
df = df[df["CC"] == "PROXY"]

# Bin the data (e.g., 0–1 sec, 1–2 sec, ..., or choose bin width)
bin_edges = np.arange(0, df["Time_to_CnC"].max() + 1, 1)  # 1-second bins
df['Bin'] = pd.cut(df["Time_to_CnC"], bins=bin_edges, right=False)

# Count entries per bin
duration_ranges = df['Bin'].value_counts().sort_index().reset_index()
duration_ranges.columns = ['Range', 'Count']

# Extract upper edge of each range for plotting
duration_ranges['Range End'] = duration_ranges['Range'].apply(lambda x: x.right)

# Cumulative percentage
duration_ranges['Cumulative Count'] = duration_ranges['Count'].cumsum()
total_count = duration_ranges['Cumulative Count'].iloc[-1]
duration_ranges['Cumulative Percentage'] = (duration_ranges['Cumulative Count'] / total_count) * 100

# Plot CDF
plt.figure(figsize=(6, 3.5))
plt.step(duration_ranges['Range End'], duration_ranges['Cumulative Percentage'],
         where='post', label='CDF (in %)', color='blue', linewidth=1.75)

plt.xlabel('Time Duration of Connections (in Sec)', fontsize=14, fontweight='bold')
plt.ylabel('Cumulative Count (in %)', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.tight_layout()

# Save
plt.savefig("<path-to-directory>")
plt.show()